# MOL_LAB DIDACTIC v0.3

**Molekülwerkstatt für den Chemieunterricht der SEK2 – mit kuratierten Vergleichspaaren**

Ziel: Von der Molekülstruktur zu Stoffeigenschaften schließen.

Didaktischer Ablauf: **Beobachten → Vermuten → Antwort prüfen → Erklärung anzeigen**.

Diese Version nutzt bewusst nur schnelle, robuste Open-Source-Bausteine: RDKit, py3Dmol, ipywidgets und pandas. xTB/CREST bleiben im großen MOL_LAB.

> **v0.2:** kompakteres Layout mit Eigenschaften rechts neben 2D/3D, kompaktere Isomeren-Vergleichskarten und zusätzliche Sicherheit zum Neuaufbau der 3D-Anzeige.


> **v0.2:** alternative schlanke 3Dmol.js-Engine ergänzt, um Colab-Rendering-Probleme nach vielen 3D-Aufrufen abzufedern.


> **v0.3:** Modul *Polarität und Löslichkeit* enthält nun kuratierte Vergleichspaare mit Mini-Aufgaben und optionalem experimentellem Bezug. Die 3D-Darstellung wurde auf vier direkte Schaltflächen umgestellt.

## 1. Installation

In Google Colab: diese Zelle einmal ausführen. Die Installation dauert normalerweise unter einer Minute.


In [ ]:
# @title 🔧 Installation ausführen { display-mode: "form" }
!pip -q install rdkit py3Dmol ipywidgets pandas


## 2. Datenbank und Funktionen laden

Diese Zelle startet die eigentliche Oberfläche. In Colab ist sie als **Form-Zelle** vorbereitet; bei Bedarf kann über **Code anzeigen/ausblenden** zwischen Bedienoberfläche und technischem Code umgeschaltet werden.


In [ ]:
# @title 🧪 MOL_LAB DIDACTIC starten { display-mode: "form" }
import json, math, textwrap, html, base64, uuid, gc
from io import BytesIO
import pandas as pd
from IPython.display import display, HTML, clear_output, Javascript
import ipywidgets as widgets
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, Descriptors, rdMolDescriptors, Crippen
from rdkit.Chem.Draw import IPythonConsole
import py3Dmol

MOLECULES = {'Polarität und Löslichkeit': [{'name': 'Wasser',
                                'smiles': 'O',
                                'niveau': 'Basis',
                                'leitfrage': 'Warum ist Wasser ein stark polares Lösungsmittel?',
                                'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                               'Drehe anschließend das 3D-Modell.',
                                               'Vergleiche die Deskriptoren mit der Leitfrage.'],
                                'frage': 'Welche Aussage passt am besten?',
                                'antworten': ['Wasser kann Wasserstoffbrücken spenden und aufnehmen.',
                                              'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                              'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                              'vorkommt.',
                                              'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                                'richtig': 0,
                                'feedback': 'Richtig: Wasser kann Wasserstoffbrücken spenden und aufnehmen.',
                                'erklaerung': 'Wasser besitzt polare O-H-Bindungen und eine gewinkelte Struktur; '
                                              'dadurch heben sich die Dipole nicht auf.',
                                'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und '
                                                'Kontext können dadurch nicht vollständig ersetzt werden.'},
                               {'name': 'Methan',
                                'smiles': 'C',
                                'niveau': 'Basis',
                                'leitfrage': 'Warum ist Methan unpolar und schlecht wasserlöslich?',
                                'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                               'Drehe anschließend das 3D-Modell.',
                                               'Vergleiche die Deskriptoren mit der Leitfrage.'],
                                'frage': 'Welche Aussage passt am besten?',
                                'antworten': ['Methan ist weitgehend unpolar und kann keine Wasserstoffbrücken bilden.',
                                              'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                              'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                              'vorkommt.',
                                              'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                                'richtig': 0,
                                'feedback': 'Richtig: Methan ist weitgehend unpolar und kann keine Wasserstoffbrücken '
                                            'bilden.',
                                'erklaerung': 'Methan ist sehr symmetrisch und besitzt keine stark polare funktionelle '
                                              'Gruppe.',
                                'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und '
                                                'Kontext können dadurch nicht vollständig ersetzt werden.'},
                               {'name': 'Ammoniak',
                                'smiles': 'N',
                                'niveau': 'Basis',
                                'leitfrage': 'Warum ist Ammoniak polar und gut wasserlöslich?',
                                'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                               'Drehe anschließend das 3D-Modell.',
                                               'Vergleiche die Deskriptoren mit der Leitfrage.'],
                                'frage': 'Welche Aussage passt am besten?',
                                'antworten': ['Ammoniak kann Wasserstoffbrücken bilden.',
                                              'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                              'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                              'vorkommt.',
                                              'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                                'richtig': 0,
                                'feedback': 'Richtig: Ammoniak kann Wasserstoffbrücken bilden.',
                                'erklaerung': 'Ammoniak besitzt polare N-H-Bindungen und ein freies Elektronenpaar am '
                                              'Stickstoff.',
                                'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und '
                                                'Kontext können dadurch nicht vollständig ersetzt werden.'},
                               {'name': 'Kohlenstoffdioxid',
                                'smiles': 'O=C=O',
                                'niveau': 'Basis',
                                'leitfrage': 'Warum ist CO₂ trotz polarer C=O-Bindungen insgesamt kaum polar?',
                                'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                               'Drehe anschließend das 3D-Modell.',
                                               'Vergleiche die Deskriptoren mit der Leitfrage.'],
                                'frage': 'Welche Aussage passt am besten?',
                                'antworten': ['Die lineare symmetrische Struktur hebt die Bindungsdipole auf.',
                                              'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                              'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                              'vorkommt.',
                                              'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                                'richtig': 0,
                                'feedback': 'Richtig: Die lineare symmetrische Struktur hebt die Bindungsdipole auf.',
                                'erklaerung': 'CO₂ besitzt polare Bindungen, aber die lineare Anordnung führt zur '
                                              'Aufhebung der Dipole.',
                                'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und '
                                                'Kontext können dadurch nicht vollständig ersetzt werden.'},
                               {'name': 'Methanol',
                                'smiles': 'CO',
                                'niveau': 'Basis',
                                'leitfrage': 'Warum ist Methanol sehr gut mit Wasser mischbar?',
                                'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                               'Drehe anschließend das 3D-Modell.',
                                               'Vergleiche die Deskriptoren mit der Leitfrage.'],
                                'frage': 'Welche Aussage passt am besten?',
                                'antworten': ['Die OH-Gruppe kann Wasserstoffbrücken bilden und der unpolare Rest ist '
                                              'klein.',
                                              'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                              'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                              'vorkommt.',
                                              'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                                'richtig': 0,
                                'feedback': 'Richtig: Die OH-Gruppe kann Wasserstoffbrücken bilden und der unpolare '
                                            'Rest ist klein.',
                                'erklaerung': 'Methanol besitzt eine polare OH-Gruppe und nur eine kleine '
                                              'Methylgruppe.',
                                'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und '
                                                'Kontext können dadurch nicht vollständig ersetzt werden.'},
                               {'name': 'Ethanol',
                                'smiles': 'CCO',
                                'niveau': 'Basis',
                                'leitfrage': 'Warum ist Ethanol mit Wasser mischbar, obwohl es auch einen unpolaren '
                                             'Teil besitzt?',
                                'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                               'Drehe anschließend das 3D-Modell.',
                                               'Vergleiche die Deskriptoren mit der Leitfrage.'],
                                'frage': 'Welche Aussage passt am besten?',
                                'antworten': ['Die OH-Gruppe kann Wasserstoffbrücken mit Wasser bilden.',
                                              'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                              'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                              'vorkommt.',
                                              'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                                'richtig': 0,
                                'feedback': 'Richtig: Die OH-Gruppe kann Wasserstoffbrücken mit Wasser bilden.',
                                'erklaerung': 'Ethanol besitzt eine polare OH-Gruppe und einen noch kleinen unpolaren '
                                              'Ethylrest.',
                                'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und '
                                                'Kontext können dadurch nicht vollständig ersetzt werden.'},
                               {'name': 'n-Butanol',
                                'smiles': 'CCCCO',
                                'niveau': 'Basis',
                                'leitfrage': 'Warum ist n-Butanol schlechter wasserlöslich als Methanol oder Ethanol?',
                                'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                               'Drehe anschließend das 3D-Modell.',
                                               'Vergleiche die Deskriptoren mit der Leitfrage.'],
                                'frage': 'Welche Aussage passt am besten?',
                                'antworten': ['Der größere unpolare Kohlenwasserstoffrest verringert die '
                                              'Wasserlöslichkeit.',
                                              'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                              'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                              'vorkommt.',
                                              'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                                'richtig': 0,
                                'feedback': 'Richtig: Der größere unpolare Kohlenwasserstoffrest verringert die '
                                            'Wasserlöslichkeit.',
                                'erklaerung': 'Die OH-Gruppe bleibt polar, aber mit wachsender Alkylkette nimmt der '
                                              'unpolare Anteil zu.',
                                'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und '
                                                'Kontext können dadurch nicht vollständig ersetzt werden.'},
                               {'name': 'Methanal',
                                'smiles': 'C=O',
                                'niveau': 'Basis',
                                'leitfrage': 'Warum ist Methanal polar, obwohl es keine OH-Gruppe besitzt?',
                                'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                               'Drehe anschließend das 3D-Modell.',
                                               'Vergleiche die Deskriptoren mit der Leitfrage.'],
                                'frage': 'Welche Aussage passt am besten?',
                                'antworten': ['Die Carbonylgruppe ist polar und kann H-Brücken aufnehmen, aber nicht '
                                              'spenden.',
                                              'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                              'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                              'vorkommt.',
                                              'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                                'richtig': 0,
                                'feedback': 'Richtig: Die Carbonylgruppe ist polar und kann H-Brücken aufnehmen, aber '
                                            'nicht spenden.',
                                'erklaerung': 'Die C=O-Doppelbindung ist polar; der Sauerstoff kann als Akzeptor '
                                              'wirken.',
                                'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und '
                                                'Kontext können dadurch nicht vollständig ersetzt werden.'},
                               {'name': 'Aceton',
                                'smiles': 'CC(=O)C',
                                'niveau': 'Basis',
                                'leitfrage': 'Warum ist Aceton gut mit Wasser mischbar, obwohl es keine OH-Gruppe '
                                             'besitzt?',
                                'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                               'Drehe anschließend das 3D-Modell.',
                                               'Vergleiche die Deskriptoren mit der Leitfrage.'],
                                'frage': 'Welche Aussage passt am besten?',
                                'antworten': ['Die C=O-Gruppe kann H-Brücken von Wasser aufnehmen.',
                                              'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                              'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                              'vorkommt.',
                                              'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                                'richtig': 0,
                                'feedback': 'Richtig: Die C=O-Gruppe kann H-Brücken von Wasser aufnehmen.',
                                'erklaerung': 'Aceton besitzt eine polare Carbonylgruppe und zwei kleine unpolare '
                                              'Methylgruppen.',
                                'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und '
                                                'Kontext können dadurch nicht vollständig ersetzt werden.'},
                               {'name': 'Diethylether',
                                'smiles': 'CCOCC',
                                'niveau': 'Basis',
                                'leitfrage': 'Warum unterscheidet sich Diethylether deutlich von Alkoholen?',
                                'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                               'Drehe anschließend das 3D-Modell.',
                                               'Vergleiche die Deskriptoren mit der Leitfrage.'],
                                'frage': 'Welche Aussage passt am besten?',
                                'antworten': ['Ether können H-Brücken aufnehmen, aber nicht spenden.',
                                              'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                              'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                              'vorkommt.',
                                              'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                                'richtig': 0,
                                'feedback': 'Richtig: Ether können H-Brücken aufnehmen, aber nicht spenden.',
                                'erklaerung': 'Diethylether enthält ein O-Atom, aber keine O-H-Bindung.',
                                'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und '
                                                'Kontext können dadurch nicht vollständig ersetzt werden.'},
                               {'name': 'Essigsäure',
                                'smiles': 'CC(=O)O',
                                'niveau': 'Basis',
                                'leitfrage': 'Warum ist Essigsäure stark polar und gut wasserlöslich?',
                                'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                               'Drehe anschließend das 3D-Modell.',
                                               'Vergleiche die Deskriptoren mit der Leitfrage.'],
                                'frage': 'Welche Aussage passt am besten?',
                                'antworten': ['Die Carboxylgruppe ist polar und kann H-Brücken bilden.',
                                              'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                              'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                              'vorkommt.',
                                              'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                                'richtig': 0,
                                'feedback': 'Richtig: Die Carboxylgruppe ist polar und kann H-Brücken bilden.',
                                'erklaerung': 'Essigsäure besitzt eine Carboxylgruppe mit C=O und O-H am selben '
                                              'Kohlenstoffatom.',
                                'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und '
                                                'Kontext können dadurch nicht vollständig ersetzt werden.'},
                               {'name': 'Schwefelwasserstoff',
                                'smiles': 'S',
                                'niveau': 'Vergleich',
                                'leitfrage': 'Warum ist Schwefelwasserstoff bei Raumtemperatur gasförmig, Wasser aber '
                                             'flüssig?',
                                'beobachten': ['Vergleiche die Molekülform mit Wasser.',
                                               'Achte darauf, ob starke Wasserstoffbrücken möglich sind.',
                                               'Nutze die Modellwerte nur als grobe Orientierung.'],
                                'frage': 'Welche Aussage passt am besten?',
                                'antworten': ['Schwefelwasserstoff bildet ähnlich starke Wasserstoffbrücken wie '
                                              'Wasser.',
                                              'Schwefelwasserstoff ist linear und deshalb unpolar.',
                                              'Schwefelwasserstoff kann deutlich schwächere zwischenmolekulare '
                                              'Wechselwirkungen ausbilden als Wasser.',
                                              'Schwefelwasserstoff ist ein Salz.'],
                                'richtig': 2,
                                'feedback': 'Richtig: H₂S ist zwar gewinkelt, bildet aber keine so starken '
                                            'Wasserstoffbrücken-Netzwerke wie Wasser.',
                                'erklaerung': 'Wasser und Schwefelwasserstoff wirken auf den ersten Blick ähnlich, '
                                              'unterscheiden sich aber stark in ihren zwischenmolekularen Kräften. '
                                              'Wasser bildet ein starkes Wasserstoffbrücken-Netzwerk; bei H₂S sind '
                                              'diese Wechselwirkungen viel schwächer.',
                                'modellgrenze': 'Für sehr kleine anorganische Moleküle sind RDKit-Deskriptoren wie '
                                                'logP/TPSA nur eingeschränkt aussagekräftig. Hier ist der '
                                                'experimentelle Bezug besonders wichtig.'},
                               {'name': 'Hexan',
                                'smiles': 'CCCCCC',
                                'niveau': 'Vergleich',
                                'leitfrage': 'Warum ist Hexan flüssig, Methan aber gasförmig, obwohl beide unpolar '
                                             'sind?',
                                'beobachten': ['Vergleiche die Molekülgröße.',
                                               'Betrachte die unpolare Oberfläche.',
                                               'Überlege, warum größere unpolare Moleküle stärkere London-Kräfte '
                                               'besitzen.'],
                                'frage': 'Welche Aussage erklärt den Unterschied am besten?',
                                'antworten': ['Hexan ist ionisch.',
                                              'Hexan ist größer und besitzt stärkere London-Dispersionskräfte.',
                                              'Methan besitzt starke Wasserstoffbrücken.',
                                              'Die Summenformel allein erklärt den Aggregatzustand.'],
                                'richtig': 1,
                                'feedback': 'Richtig: Größere unpolare Moleküle besitzen stärkere '
                                            'London-Dispersionskräfte.',
                                'erklaerung': 'Unpolar bedeutet nicht wechselwirkungsfrei. Hexan hat eine deutlich '
                                              'größere Oberfläche als Methan; dadurch wirken stärkere London-Kräfte '
                                              'zwischen den Molekülen.',
                                'modellgrenze': 'Die App zeigt London-Kräfte nicht direkt, sondern nur indirekt über '
                                                'Molekülgröße und Oberfläche.'},
                               {'name': '1-Hexanol',
                                'smiles': 'CCCCCCO',
                                'niveau': 'Vergleich',
                                'leitfrage': 'Warum ist 1-Hexanol viel schlechter wasserlöslich als Ethanol?',
                                'beobachten': ['Suche die OH-Gruppe.',
                                               'Vergleiche den unpolaren Kohlenwasserstoffrest mit Ethanol.',
                                               'Achte auf logP, TPSA und die Wasserlöslichkeits-Tendenz.'],
                                'frage': 'Welche Aussage passt am besten?',
                                'antworten': ['1-Hexanol besitzt keine polare Gruppe.',
                                              'Die OH-Gruppe ist polar, aber der große unpolare Rest verringert die '
                                              'Wasserlöslichkeit.',
                                              '1-Hexanol ist ein Salz.',
                                              'Alle Alkohole sind vollständig wasserlöslich.'],
                                'richtig': 1,
                                'feedback': 'Richtig: Die OH-Gruppe bleibt polar, aber der unpolare Rest wird mit '
                                            'zunehmender Kettenlänge wichtiger.',
                                'erklaerung': 'Bei kurzen Alkoholen dominiert die OH-Gruppe die Wechselwirkung mit '
                                              'Wasser. Bei 1-Hexanol ist der unpolare Kohlenwasserstoffrest so groß, '
                                              'dass die Wasserlöslichkeit deutlich sinkt.',
                                'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und '
                                                'Kontext können dadurch nicht vollständig ersetzt werden.'},
                               {'name': 'Palmitinsäure',
                                'smiles': 'CCCCCCCCCCCCCCCC(=O)O',
                                'niveau': 'Vertiefung',
                                'leitfrage': 'Warum ist Palmitinsäure trotz Carboxylgruppe kaum wasserlöslich?',
                                'beobachten': ['Suche die Carboxylgruppe.',
                                               'Vergleiche die Länge des unpolaren Restes mit Essigsäure.',
                                               'Überlege, welcher Molekülteil die Stoffeigenschaft stärker prägt.'],
                                'frage': 'Welche Aussage passt am besten?',
                                'antworten': ['Alle Carbonsäuren sind gut wasserlöslich.',
                                              'Die lange unpolare Kohlenwasserstoffkette überlagert die Wirkung der '
                                              'polaren Carboxylgruppe.',
                                              'Palmitinsäure besitzt keine polare Gruppe.',
                                              'Palmitinsäure ist kleiner als Essigsäure.'],
                                'richtig': 1,
                                'feedback': 'Richtig: Die große unpolare Kette bestimmt die Wasserlöslichkeit '
                                            'wesentlich mit.',
                                'erklaerung': 'Palmitinsäure besitzt zwar eine Carboxylgruppe, aber auch eine sehr '
                                              'lange unpolare Kohlenwasserstoffkette. Dadurch ist sie in Wasser kaum '
                                              'löslich.',
                                'modellgrenze': 'Das Modell berücksichtigt keine Kristallpackung, pH-Effekte oder die '
                                                'Bildung von Aggregaten/Mizellen.'},
                               {'name': 'Diethylenglycol',
                                'smiles': 'OCCOCCO',
                                'niveau': 'Vergleich',
                                'leitfrage': 'Warum ist Diethylenglycol deutlich wasserfreundlicher als Diethylether?',
                                'beobachten': ['Zähle die Sauerstoffatome.',
                                               'Suche OH-Gruppen.',
                                               'Vergleiche H-Brücken-Donoren und -Akzeptoren.'],
                                'frage': 'Welche Aussage passt am besten?',
                                'antworten': ['Diethylenglycol besitzt mehrere Möglichkeiten für Wasserstoffbrücken.',
                                              'Diethylenglycol ist völlig unpolar.',
                                              'Diethylether kann mehr H-Brücken spenden.',
                                              'Beide Stoffe müssen wegen der Ethergruppe identisch löslich sein.'],
                                'richtig': 0,
                                'feedback': 'Richtig: Diethylenglycol besitzt zwei OH-Gruppen und mehrere O-Atome.',
                                'erklaerung': 'Diethylether kann H-Brücken nur aufnehmen. Diethylenglycol kann durch '
                                              'seine OH-Gruppen auch H-Brücken spenden und besitzt eine größere polare '
                                              'Oberfläche.',
                                'modellgrenze': 'Die Toxizität von Diethylenglycol ist hier nicht Thema; betrachtet '
                                                'wird nur der Struktur–Eigenschaft-Zusammenhang.'}],
 'Isomerie': [{'name': 'Ethanol',
               'smiles': 'CCO',
               'niveau': 'Basis',
               'leitfrage': 'Warum hat Ethanol andere Eigenschaften als Dimethylether, obwohl beide C₂H₆O besitzen?',
               'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                              'Drehe anschließend das 3D-Modell.',
                              'Vergleiche die Deskriptoren mit der Leitfrage.'],
               'frage': 'Welche Aussage passt am besten?',
               'antworten': ['Ethanol besitzt eine OH-Gruppe und kann H-Brücken spenden.',
                             'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                             'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.',
                             'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
               'richtig': 0,
               'feedback': 'Richtig: Ethanol besitzt eine OH-Gruppe und kann H-Brücken spenden.',
               'erklaerung': 'Gleiche Summenformel, aber andere Verknüpfung der Atome führt zu anderen funktionellen '
                             'Gruppen.',
               'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können '
                               'dadurch nicht vollständig ersetzt werden.'},
              {'name': 'Dimethylether',
               'smiles': 'COC',
               'niveau': 'Basis',
               'leitfrage': 'Warum siedet Dimethylether viel niedriger als Ethanol?',
               'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                              'Drehe anschließend das 3D-Modell.',
                              'Vergleiche die Deskriptoren mit der Leitfrage.'],
               'frage': 'Welche Aussage passt am besten?',
               'antworten': ['Dimethylether kann keine Wasserstoffbrücken spenden.',
                             'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                             'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.',
                             'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
               'richtig': 0,
               'feedback': 'Richtig: Dimethylether kann keine Wasserstoffbrücken spenden.',
               'erklaerung': 'Er besitzt eine Ethergruppe statt einer Alkoholgruppe.',
               'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können '
                               'dadurch nicht vollständig ersetzt werden.'},
              {'name': 'Propanal',
               'smiles': 'CCC=O',
               'niveau': 'Basis',
               'leitfrage': 'Wie unterscheidet sich Propanal von Aceton?',
               'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                              'Drehe anschließend das 3D-Modell.',
                              'Vergleiche die Deskriptoren mit der Leitfrage.'],
               'frage': 'Welche Aussage passt am besten?',
               'antworten': ['Propanal ist ein Aldehyd.',
                             'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                             'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.',
                             'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
               'richtig': 0,
               'feedback': 'Richtig: Propanal ist ein Aldehyd.',
               'erklaerung': 'Die Carbonylgruppe liegt am Ende der Kette.',
               'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können '
                               'dadurch nicht vollständig ersetzt werden.'},
              {'name': 'Aceton',
               'smiles': 'CC(=O)C',
               'niveau': 'Basis',
               'leitfrage': 'Wie unterscheidet sich Aceton von Propanal?',
               'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                              'Drehe anschließend das 3D-Modell.',
                              'Vergleiche die Deskriptoren mit der Leitfrage.'],
               'frage': 'Welche Aussage passt am besten?',
               'antworten': ['Aceton ist ein Keton.',
                             'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                             'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.',
                             'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
               'richtig': 0,
               'feedback': 'Richtig: Aceton ist ein Keton.',
               'erklaerung': 'Die Carbonylgruppe liegt zwischen zwei Kohlenstoffatomen.',
               'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können '
                               'dadurch nicht vollständig ersetzt werden.'},
              {'name': 'Butan',
               'smiles': 'CCCC',
               'niveau': 'Basis',
               'leitfrage': 'Was unterscheidet Butan von Isobutan?',
               'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                              'Drehe anschließend das 3D-Modell.',
                              'Vergleiche die Deskriptoren mit der Leitfrage.'],
               'frage': 'Welche Aussage passt am besten?',
               'antworten': ['Butan ist unverzweigt, Isobutan ist verzweigt.',
                             'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                             'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.',
                             'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
               'richtig': 0,
               'feedback': 'Richtig: Butan ist unverzweigt, Isobutan ist verzweigt.',
               'erklaerung': 'Beide haben dieselbe Summenformel, aber unterschiedliche Kohlenstoffverknüpfung.',
               'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können '
                               'dadurch nicht vollständig ersetzt werden.'},
              {'name': 'Isobutan',
               'smiles': 'CC(C)C',
               'niveau': 'Basis',
               'leitfrage': 'Warum ist Isobutan kompakter als Butan?',
               'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                              'Drehe anschließend das 3D-Modell.',
                              'Vergleiche die Deskriptoren mit der Leitfrage.'],
               'frage': 'Welche Aussage passt am besten?',
               'antworten': ['Die Kohlenstoffkette ist verzweigt.',
                             'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                             'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.',
                             'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
               'richtig': 0,
               'feedback': 'Richtig: Die Kohlenstoffkette ist verzweigt.',
               'erklaerung': 'Verzweigung verändert Molekülform und damit Stoffeigenschaften.',
               'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können '
                               'dadurch nicht vollständig ersetzt werden.'},
              {'name': 'Propansäure',
               'smiles': 'CCC(=O)O',
               'niveau': 'Basis',
               'leitfrage': 'Warum unterscheidet sich Propansäure stark von Essigsäuremethylester?',
               'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                              'Drehe anschließend das 3D-Modell.',
                              'Vergleiche die Deskriptoren mit der Leitfrage.'],
               'frage': 'Welche Aussage passt am besten?',
               'antworten': ['Propansäure enthält eine Carboxylgruppe.',
                             'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                             'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.',
                             'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
               'richtig': 0,
               'feedback': 'Richtig: Propansäure enthält eine Carboxylgruppe.',
               'erklaerung': 'Funktionelle Isomerie: Carbonsäure vs. Ester.',
               'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können '
                               'dadurch nicht vollständig ersetzt werden.'},
              {'name': 'Essigsäuremethylester',
               'smiles': 'CC(=O)OC',
               'niveau': 'Basis',
               'leitfrage': 'Warum ist Essigsäuremethylester kein Säure-Isomer im Eigenschaftssinn?',
               'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                              'Drehe anschließend das 3D-Modell.',
                              'Vergleiche die Deskriptoren mit der Leitfrage.'],
               'frage': 'Welche Aussage passt am besten?',
               'antworten': ['Er enthält eine Estergruppe und keine Carboxyl-OH-Gruppe.',
                             'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                             'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.',
                             'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
               'richtig': 0,
               'feedback': 'Richtig: Er enthält eine Estergruppe und keine Carboxyl-OH-Gruppe.',
               'erklaerung': 'Gleiche Summenformel kann durch andere funktionelle Gruppe ganz andere Eigenschaften '
                             'ergeben.',
               'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können '
                               'dadurch nicht vollständig ersetzt werden.'},
              {'name': 'D-Alanin',
               'smiles': 'C[C@@H](N)C(=O)O',
               'niveau': 'Basis',
               'leitfrage': 'Warum sind D- und L-Alanin trotz gleicher Verknüpfung nicht dasselbe Molekül?',
               'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                              'Drehe anschließend das 3D-Modell.',
                              'Vergleiche die Deskriptoren mit der Leitfrage.'],
               'frage': 'Welche Aussage passt am besten?',
               'antworten': ['Sie sind Enantiomere, also spiegelbildliche Moleküle.',
                             'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                             'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.',
                             'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
               'richtig': 0,
               'feedback': 'Richtig: Sie sind Enantiomere, also spiegelbildliche Moleküle.',
               'erklaerung': 'Chiralität zeigt, dass auch die räumliche Anordnung entscheidend ist.',
               'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können '
                               'dadurch nicht vollständig ersetzt werden.'},
              {'name': 'L-Alanin',
               'smiles': 'C[C@H](N)C(=O)O',
               'niveau': 'Basis',
               'leitfrage': 'Warum ist Chiralität bei Aminosäuren biologisch wichtig?',
               'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                              'Drehe anschließend das 3D-Modell.',
                              'Vergleiche die Deskriptoren mit der Leitfrage.'],
               'frage': 'Welche Aussage passt am besten?',
               'antworten': ['L-Alanin ist eine spiegelbildliche Form von D-Alanin.',
                             'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                             'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.',
                             'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
               'richtig': 0,
               'feedback': 'Richtig: L-Alanin ist eine spiegelbildliche Form von D-Alanin.',
               'erklaerung': 'Biologische Systeme unterscheiden Spiegelbilder häufig sehr genau.',
               'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können '
                               'dadurch nicht vollständig ersetzt werden.'},
              {'name': 'Glucose offen',
               'smiles': 'O=CC(O)C(O)C(O)C(O)CO',
               'niveau': 'Basis',
               'leitfrage': 'Warum ist Glucose ein komplexeres Isomeriebeispiel?',
               'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                              'Drehe anschließend das 3D-Modell.',
                              'Vergleiche die Deskriptoren mit der Leitfrage.'],
               'frage': 'Welche Aussage passt am besten?',
               'antworten': ['Offenkettige Glucose ist eine Aldose.',
                             'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                             'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.',
                             'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
               'richtig': 0,
               'feedback': 'Richtig: Offenkettige Glucose ist eine Aldose.',
               'erklaerung': 'Glucose und Fructose haben dieselbe Summenformel, unterscheiden sich aber in der '
                             'Carbonylgruppe.',
               'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können '
                               'dadurch nicht vollständig ersetzt werden.'},
              {'name': 'Fructose offen',
               'smiles': 'OCC(O)C(O)C(O)C(=O)CO',
               'niveau': 'Basis',
               'leitfrage': 'Warum ist Fructose trotz gleicher Summenformel anders als Glucose?',
               'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                              'Drehe anschließend das 3D-Modell.',
                              'Vergleiche die Deskriptoren mit der Leitfrage.'],
               'frage': 'Welche Aussage passt am besten?',
               'antworten': ['Fructose ist eine Ketose.',
                             'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                             'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.',
                             'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
               'richtig': 0,
               'feedback': 'Richtig: Fructose ist eine Ketose.',
               'erklaerung': 'In Lösung sind zusätzlich Ringformen wichtig; hier wird bewusst vereinfacht.',
               'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können '
                               'dadurch nicht vollständig ersetzt werden.'},
              {'name': 'Maleinsäure',
               'smiles': 'O=C(O)/C=C\\C(=O)O',
               'niveau': 'Vertiefung',
               'leitfrage': 'Warum kann die cis/trans-Anordnung die Stoffeigenschaften stark verändern?',
               'beobachten': ['Vergleiche die Lage der beiden Carboxylgruppen.',
                              'Achte auf die räumliche Nähe der Gruppen.',
                              'Vergleiche mit Fumarsäure im Isomerenvergleich.'],
               'frage': 'Welche Aussage passt am besten?',
               'antworten': ['Maleinsäure und Fumarsäure unterscheiden sich in der räumlichen Anordnung an der '
                             'Doppelbindung.',
                             'Sie haben unterschiedliche Summenformeln.',
                             'Bei Doppelbindungen ist räumliche Anordnung unmöglich.',
                             'Die Carboxylgruppen spielen keine Rolle.'],
               'richtig': 0,
               'feedback': 'Richtig: Es handelt sich um cis/trans-Isomerie an der C=C-Doppelbindung.',
               'erklaerung': 'Maleinsäure ist das cis-Isomer; die Carboxylgruppen liegen räumlich nahe beieinander. '
                             'Dadurch unterscheiden sich Eigenschaften wie Löslichkeit, Schmelzpunkt und mögliche '
                             'intramolekulare Wechselwirkungen von Fumarsäure.',
               'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können '
                               'dadurch nicht vollständig ersetzt werden.'},
              {'name': 'Fumarsäure',
               'smiles': 'O=C(O)/C=C/C(=O)O',
               'niveau': 'Vertiefung',
               'leitfrage': 'Warum ist Fumarsäure trotz gleicher Summenformel anders als Maleinsäure?',
               'beobachten': ['Vergleiche die Lage der Carboxylgruppen.',
                              'Achte auf die trans-Anordnung.',
                              'Überlege, warum das die Wechselwirkungen verändert.'],
               'frage': 'Welche Aussage passt am besten?',
               'antworten': ['Fumarsäure ist das trans-Isomer von Maleinsäure.',
                             'Fumarsäure hat keine Carboxylgruppen.',
                             'Fumarsäure und Maleinsäure sind identische Moleküle.',
                             'Fumarsäure ist ein Alkan.'],
               'richtig': 0,
               'feedback': 'Richtig: Fumarsäure ist das trans-Isomer.',
               'erklaerung': 'Fumarsäure besitzt dieselbe Summenformel wie Maleinsäure, aber eine andere räumliche '
                             'Anordnung. Diese scheinbar kleine Änderung kann deutliche Stoffeigenschaftsunterschiede '
                             'bewirken.',
               'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können '
                               'dadurch nicht vollständig ersetzt werden.'}],
 'Funktionelle Gruppen': [{'name': 'Methanol',
                           'smiles': 'CO',
                           'niveau': 'Basis',
                           'leitfrage': 'Woran erkennt man einen Alkohol?',
                           'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                          'Drehe anschließend das 3D-Modell.',
                                          'Vergleiche die Deskriptoren mit der Leitfrage.'],
                           'frage': 'Welche Aussage passt am besten?',
                           'antworten': ['Methanol besitzt eine Alkoholgruppe.',
                                         'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                         'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                         'vorkommt.',
                                         'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                           'richtig': 0,
                           'feedback': 'Richtig: Methanol besitzt eine Alkoholgruppe.',
                           'erklaerung': 'Eine OH-Gruppe an einem gesättigten Kohlenstoff prägt Polarität und '
                                         'H-Brücken.',
                           'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext '
                                           'können dadurch nicht vollständig ersetzt werden.'},
                          {'name': 'Ethanal',
                           'smiles': 'CC=O',
                           'niveau': 'Basis',
                           'leitfrage': 'Woran erkennt man einen Aldehyd?',
                           'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                          'Drehe anschließend das 3D-Modell.',
                                          'Vergleiche die Deskriptoren mit der Leitfrage.'],
                           'frage': 'Welche Aussage passt am besten?',
                           'antworten': ['Ethanal besitzt eine Aldehydgruppe.',
                                         'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                         'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                         'vorkommt.',
                                         'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                           'richtig': 0,
                           'feedback': 'Richtig: Ethanal besitzt eine Aldehydgruppe.',
                           'erklaerung': 'Die Carbonylgruppe liegt am Kettenende.',
                           'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext '
                                           'können dadurch nicht vollständig ersetzt werden.'},
                          {'name': 'Aceton',
                           'smiles': 'CC(=O)C',
                           'niveau': 'Basis',
                           'leitfrage': 'Woran erkennt man ein Keton?',
                           'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                          'Drehe anschließend das 3D-Modell.',
                                          'Vergleiche die Deskriptoren mit der Leitfrage.'],
                           'frage': 'Welche Aussage passt am besten?',
                           'antworten': ['Aceton besitzt eine Ketogruppe.',
                                         'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                         'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                         'vorkommt.',
                                         'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                           'richtig': 0,
                           'feedback': 'Richtig: Aceton besitzt eine Ketogruppe.',
                           'erklaerung': 'Die Carbonylgruppe liegt zwischen zwei Kohlenstoffresten.',
                           'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext '
                                           'können dadurch nicht vollständig ersetzt werden.'},
                          {'name': 'Essigsäure',
                           'smiles': 'CC(=O)O',
                           'niveau': 'Basis',
                           'leitfrage': 'Woran erkennt man eine Carbonsäure?',
                           'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                          'Drehe anschließend das 3D-Modell.',
                                          'Vergleiche die Deskriptoren mit der Leitfrage.'],
                           'frage': 'Welche Aussage passt am besten?',
                           'antworten': ['Essigsäure besitzt eine Carboxylgruppe.',
                                         'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                         'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                         'vorkommt.',
                                         'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                           'richtig': 0,
                           'feedback': 'Richtig: Essigsäure besitzt eine Carboxylgruppe.',
                           'erklaerung': 'C=O und OH am selben Kohlenstoffatom bilden die Carboxylgruppe.',
                           'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext '
                                           'können dadurch nicht vollständig ersetzt werden.'},
                          {'name': 'Ethylacetat',
                           'smiles': 'CC(=O)OCC',
                           'niveau': 'Basis',
                           'leitfrage': 'Woran erkennt man einen Ester?',
                           'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                          'Drehe anschließend das 3D-Modell.',
                                          'Vergleiche die Deskriptoren mit der Leitfrage.'],
                           'frage': 'Welche Aussage passt am besten?',
                           'antworten': ['Ethylacetat besitzt eine Estergruppe.',
                                         'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                         'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                         'vorkommt.',
                                         'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                           'richtig': 0,
                           'feedback': 'Richtig: Ethylacetat besitzt eine Estergruppe.',
                           'erklaerung': 'Das Strukturmotiv ist C(=O)-O-C.',
                           'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext '
                                           'können dadurch nicht vollständig ersetzt werden.'},
                          {'name': 'Diethylether',
                           'smiles': 'CCOCC',
                           'niveau': 'Basis',
                           'leitfrage': 'Woran erkennt man einen Ether?',
                           'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                          'Drehe anschließend das 3D-Modell.',
                                          'Vergleiche die Deskriptoren mit der Leitfrage.'],
                           'frage': 'Welche Aussage passt am besten?',
                           'antworten': ['Diethylether besitzt eine Ethergruppe.',
                                         'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                         'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                         'vorkommt.',
                                         'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                           'richtig': 0,
                           'feedback': 'Richtig: Diethylether besitzt eine Ethergruppe.',
                           'erklaerung': 'Ein Sauerstoffatom ist mit zwei Kohlenstoffresten verbunden.',
                           'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext '
                                           'können dadurch nicht vollständig ersetzt werden.'},
                          {'name': 'Acetanhydrid',
                           'smiles': 'CC(=O)OC(=O)C',
                           'niveau': 'Basis',
                           'leitfrage': 'Woran erkennt man ein Carbonsäureanhydrid?',
                           'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                          'Drehe anschließend das 3D-Modell.',
                                          'Vergleiche die Deskriptoren mit der Leitfrage.'],
                           'frage': 'Welche Aussage passt am besten?',
                           'antworten': ['Acetanhydrid besitzt eine Anhydridgruppe.',
                                         'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                         'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                         'vorkommt.',
                                         'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                           'richtig': 0,
                           'feedback': 'Richtig: Acetanhydrid besitzt eine Anhydridgruppe.',
                           'erklaerung': 'Zwei Acylgruppen sind über ein Sauerstoffatom verbunden.',
                           'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext '
                                           'können dadurch nicht vollständig ersetzt werden.'},
                          {'name': 'Ethylamin',
                           'smiles': 'CCN',
                           'niveau': 'Basis',
                           'leitfrage': 'Woran erkennt man ein Amin?',
                           'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                          'Drehe anschließend das 3D-Modell.',
                                          'Vergleiche die Deskriptoren mit der Leitfrage.'],
                           'frage': 'Welche Aussage passt am besten?',
                           'antworten': ['Ethylamin besitzt eine Aminogruppe.',
                                         'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                         'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                         'vorkommt.',
                                         'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                           'richtig': 0,
                           'feedback': 'Richtig: Ethylamin besitzt eine Aminogruppe.',
                           'erklaerung': 'Amine enthalten Stickstoff und können basische Eigenschaften zeigen.',
                           'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext '
                                           'können dadurch nicht vollständig ersetzt werden.'},
                          {'name': 'Acetamid',
                           'smiles': 'CC(=O)N',
                           'niveau': 'Basis',
                           'leitfrage': 'Woran erkennt man ein Amid?',
                           'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                          'Drehe anschließend das 3D-Modell.',
                                          'Vergleiche die Deskriptoren mit der Leitfrage.'],
                           'frage': 'Welche Aussage passt am besten?',
                           'antworten': ['Acetamid besitzt eine Amidgruppe.',
                                         'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                         'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                         'vorkommt.',
                                         'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                           'richtig': 0,
                           'feedback': 'Richtig: Acetamid besitzt eine Amidgruppe.',
                           'erklaerung': 'C(=O)-N ist das zentrale Motiv; die Peptidbindung ist ebenfalls ein Amid.',
                           'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext '
                                           'können dadurch nicht vollständig ersetzt werden.'},
                          {'name': 'Phenol',
                           'smiles': 'c1ccccc1O',
                           'niveau': 'Basis',
                           'leitfrage': 'Warum ist Phenol nicht einfach ein normaler Alkohol?',
                           'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                          'Drehe anschließend das 3D-Modell.',
                                          'Vergleiche die Deskriptoren mit der Leitfrage.'],
                           'frage': 'Welche Aussage passt am besten?',
                           'antworten': ['Die OH-Gruppe ist direkt an einen aromatischen Ring gebunden.',
                                         'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                         'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                         'vorkommt.',
                                         'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                           'richtig': 0,
                           'feedback': 'Richtig: Die OH-Gruppe ist direkt an einen aromatischen Ring gebunden.',
                           'erklaerung': 'Phenole unterscheiden sich in Eigenschaften und Reaktivität von einfachen '
                                         'Alkoholen.',
                           'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext '
                                           'können dadurch nicht vollständig ersetzt werden.'},
                          {'name': 'Anilin',
                           'smiles': 'c1ccccc1N',
                           'niveau': 'Basis',
                           'leitfrage': 'Warum ist Anilin ein aromatisches Amin?',
                           'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                          'Drehe anschließend das 3D-Modell.',
                                          'Vergleiche die Deskriptoren mit der Leitfrage.'],
                           'frage': 'Welche Aussage passt am besten?',
                           'antworten': ['Anilin besitzt eine Aminogruppe am aromatischen Ring.',
                                         'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                         'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                         'vorkommt.',
                                         'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                           'richtig': 0,
                           'feedback': 'Richtig: Anilin besitzt eine Aminogruppe am aromatischen Ring.',
                           'erklaerung': 'Aromat und Aminogruppe beeinflussen sich gegenseitig.',
                           'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext '
                                           'können dadurch nicht vollständig ersetzt werden.'},
                          {'name': 'Salicylsäure',
                           'smiles': 'O=C(O)c1ccccc1O',
                           'niveau': 'Vertiefung',
                           'leitfrage': 'Welche funktionellen Gruppen erkennt man in Salicylsäure?',
                           'beobachten': ['Suche die Carboxylgruppe.',
                                          'Suche die phenolische OH-Gruppe.',
                                          'Achte auf den aromatischen Ring.'],
                           'frage': 'Welche Kombination passt am besten?',
                           'antworten': ['Carbonsäure, Phenol und aromatischer Ring.',
                                         'Alkan und Keton.',
                                         'Ether und Amin.',
                                         'Nur eine unpolare Kohlenwasserstoffkette.'],
                           'richtig': 0,
                           'feedback': 'Richtig: Salicylsäure enthält Carbonsäure, Phenol und einen aromatischen Ring.',
                           'erklaerung': 'Salicylsäure ist ein gutes Beispiel dafür, dass ein Molekül mehrere '
                                         'funktionelle Gruppen tragen kann. Diese Gruppen bestimmen gemeinsam '
                                         'Polarität, Säureeigenschaften und Wechselwirkungen.',
                           'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext '
                                           'können dadurch nicht vollständig ersetzt werden.'},
                          {'name': 'Vanillin',
                           'smiles': 'COc1cc(C=O)ccc1O',
                           'niveau': 'Vertiefung',
                           'leitfrage': 'Warum ist Vanillin ein gutes Beispiel für mehrere funktionelle Gruppen in '
                                        'einem Alltagsmolekül?',
                           'beobachten': ['Suche die Aldehydgruppe.',
                                          'Suche die Methoxy-/Ethergruppe.',
                                          'Suche die phenolische OH-Gruppe und den aromatischen Ring.'],
                           'frage': 'Welche Aussage passt am besten?',
                           'antworten': ['Vanillin enthält Aldehyd, Ether/Methoxygruppe, Phenol und Aromat.',
                                         'Vanillin ist ein einfaches Alkan.',
                                         'Vanillin enthält keine Heteroatome.',
                                         'Vanillin ist ein Salz.'],
                           'richtig': 0,
                           'feedback': 'Richtig: Vanillin kombiniert mehrere funktionelle Gruppen in einem bekannten '
                                       'Aromastoff.',
                           'erklaerung': 'Vanillin zeigt, wie mehrere funktionelle Gruppen zusammen ein '
                                         'charakteristisches Molekül ergeben. Die Struktur erklärt, warum es polarere '
                                         'Bereiche besitzt, aber auch einen aromatischen, eher unpolaren Bereich.',
                           'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext '
                                           'können dadurch nicht vollständig ersetzt werden.'}],
 'Konformere': [{'name': 'Butan',
                 'smiles': 'CCCC',
                 'niveau': 'Basis',
                 'leitfrage': 'Warum gibt es bei Butan mehrere räumliche Formen?',
                 'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                'Drehe anschließend das 3D-Modell.',
                                'Vergleiche die Deskriptoren mit der Leitfrage.'],
                 'frage': 'Welche Aussage passt am besten?',
                 'antworten': ['Konformere entstehen durch Rotation um Einfachbindungen.',
                               'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                               'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.',
                               'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                 'richtig': 0,
                 'feedback': 'Richtig: Konformere entstehen durch Rotation um Einfachbindungen.',
                 'erklaerung': 'Butan kann um C-C-Einfachbindungen rotieren; verschiedene Formen haben verschiedene '
                               'relative Energien.',
                 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können '
                                 'dadurch nicht vollständig ersetzt werden.'},
                {'name': '1,2-Dichlorethan',
                 'smiles': 'ClCCCl',
                 'niveau': 'Basis',
                 'leitfrage': 'Wie beeinflusst Rotation die Orientierung der Chloratome?',
                 'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                'Drehe anschließend das 3D-Modell.',
                                'Vergleiche die Deskriptoren mit der Leitfrage.'],
                 'frage': 'Welche Aussage passt am besten?',
                 'antworten': ['Gruppen können im Raum unterschiedlich nahe kommen.',
                               'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                               'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.',
                               'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                 'richtig': 0,
                 'feedback': 'Richtig: Gruppen können im Raum unterschiedlich nahe kommen.',
                 'erklaerung': 'Rotation verändert Abstände und Wechselwirkungen.',
                 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können '
                                 'dadurch nicht vollständig ersetzt werden.'},
                {'name': 'Ethylenglykol',
                 'smiles': 'OCCO',
                 'niveau': 'Basis',
                 'leitfrage': 'Warum kann Ethylenglykol mehrere günstige Formen einnehmen?',
                 'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                'Drehe anschließend das 3D-Modell.',
                                'Vergleiche die Deskriptoren mit der Leitfrage.'],
                 'frage': 'Welche Aussage passt am besten?',
                 'antworten': ['Mehrere Einfachbindungen können rotieren.',
                               'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                               'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.',
                               'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                 'richtig': 0,
                 'feedback': 'Richtig: Mehrere Einfachbindungen können rotieren.',
                 'erklaerung': 'Die beiden OH-Gruppen und flexible Bindungen ermöglichen mehrere Konformere.',
                 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können '
                                 'dadurch nicht vollständig ersetzt werden.'},
                {'name': 'Cyclohexan',
                 'smiles': 'C1CCCCC1',
                 'niveau': 'Basis',
                 'leitfrage': 'Warum ist die Sesselstruktur von Cyclohexan wichtig?',
                 'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                'Drehe anschließend das 3D-Modell.',
                                'Vergleiche die Deskriptoren mit der Leitfrage.'],
                 'frage': 'Welche Aussage passt am besten?',
                 'antworten': ['Eine nicht-planare Form verringert ungünstige Spannungen.',
                               'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                               'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.',
                               'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                 'richtig': 0,
                 'feedback': 'Richtig: Eine nicht-planare Form verringert ungünstige Spannungen.',
                 'erklaerung': 'Cyclohexan ist nicht planar; die Sesselstruktur ist energetisch günstig.',
                 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können '
                                 'dadurch nicht vollständig ersetzt werden.'},
                {'name': 'Milchsäure',
                 'smiles': 'C[C@H](O)C(=O)O',
                 'niveau': 'Basis',
                 'leitfrage': 'Warum ist Milchsäure ein gutes Beispiel für Flexibilität und Chiralität?',
                 'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                'Drehe anschließend das 3D-Modell.',
                                'Vergleiche die Deskriptoren mit der Leitfrage.'],
                 'frage': 'Welche Aussage passt am besten?',
                 'antworten': ['Sie besitzt ein chirales Zentrum und mehrere polare Gruppen.',
                               'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                               'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.',
                               'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                 'richtig': 0,
                 'feedback': 'Richtig: Sie besitzt ein chirales Zentrum und mehrere polare Gruppen.',
                 'erklaerung': 'Milchsäure verbindet funktionelle Gruppen, Chiralität und Beweglichkeit.',
                 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können '
                                 'dadurch nicht vollständig ersetzt werden.'}],
 'Wirkstoffe und Biomoleküle': [{'name': 'Coffein',
                                 'smiles': 'Cn1cnc2n(C)c(=O)n(C)c(=O)c12',
                                 'niveau': 'Basis',
                                 'leitfrage': 'Warum kann Coffein mit Biomolekülen wechselwirken?',
                                 'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                                'Drehe anschließend das 3D-Modell.',
                                                'Vergleiche die Deskriptoren mit der Leitfrage.'],
                                 'frage': 'Welche Aussage passt am besten?',
                                 'antworten': ['Coffein besitzt mehrere H-Brücken-Akzeptoren, aber keine Donoren.',
                                               'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                               'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                               'vorkommt.',
                                               'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                                 'richtig': 0,
                                 'feedback': 'Richtig: Coffein besitzt mehrere H-Brücken-Akzeptoren, aber keine '
                                             'Donoren.',
                                 'erklaerung': 'Das heteroatomreiche Ringsystem erlaubt mehrere schwache '
                                               'Wechselwirkungen.',
                                 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und '
                                                 'Kontext können dadurch nicht vollständig ersetzt werden.'},
                                {'name': 'Aspirin',
                                 'smiles': 'CC(=O)Oc1ccccc1C(=O)O',
                                 'niveau': 'Basis',
                                 'leitfrage': 'Welche funktionellen Gruppen prägen Aspirin?',
                                 'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                                'Drehe anschließend das 3D-Modell.',
                                                'Vergleiche die Deskriptoren mit der Leitfrage.'],
                                 'frage': 'Welche Aussage passt am besten?',
                                 'antworten': ['Aspirin enthält Carboxylgruppe, Estergruppe und Aromat.',
                                               'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                               'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                               'vorkommt.',
                                               'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                                 'richtig': 0,
                                 'feedback': 'Richtig: Aspirin enthält Carboxylgruppe, Estergruppe und Aromat.',
                                 'erklaerung': 'Polare und unpolare Bereiche kommen in einem Wirkstoffmolekül '
                                               'zusammen.',
                                 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und '
                                                 'Kontext können dadurch nicht vollständig ersetzt werden.'},
                                {'name': 'Paracetamol',
                                 'smiles': 'CC(=O)Nc1ccc(O)cc1',
                                 'niveau': 'Basis',
                                 'leitfrage': 'Welche Strukturmerkmale machen Paracetamol polar?',
                                 'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                                'Drehe anschließend das 3D-Modell.',
                                                'Vergleiche die Deskriptoren mit der Leitfrage.'],
                                 'frage': 'Welche Aussage passt am besten?',
                                 'antworten': ['Paracetamol enthält Phenol und Amid.',
                                               'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                               'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                               'vorkommt.',
                                               'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                                 'richtig': 0,
                                 'feedback': 'Richtig: Paracetamol enthält Phenol und Amid.',
                                 'erklaerung': 'Diese Gruppen können H-Brücken bilden; der Aromat ist unpolarer.',
                                 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und '
                                                 'Kontext können dadurch nicht vollständig ersetzt werden.'},
                                {'name': 'Ibuprofen',
                                 'smiles': 'CC(C)Cc1ccc(cc1)[C@@H](C)C(=O)O',
                                 'niveau': 'Basis',
                                 'leitfrage': 'Warum ist Ibuprofen relativ lipophil?',
                                 'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                                'Drehe anschließend das 3D-Modell.',
                                                'Vergleiche die Deskriptoren mit der Leitfrage.'],
                                 'frage': 'Welche Aussage passt am besten?',
                                 'antworten': ['Es besitzt eine polare Carboxylgruppe und einen großen unpolaren '
                                               'Bereich.',
                                               'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                               'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                               'vorkommt.',
                                               'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                                 'richtig': 0,
                                 'feedback': 'Richtig: Es besitzt eine polare Carboxylgruppe und einen großen '
                                             'unpolaren Bereich.',
                                 'erklaerung': 'Wirkstoffe enthalten oft beide Bereiche: polar für Wechselwirkungen, '
                                               'unpolar für Bindungstaschen/Membranen.',
                                 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und '
                                                 'Kontext können dadurch nicht vollständig ersetzt werden.'},
                                {'name': 'Acetazolamid',
                                 'smiles': 'CC(=O)Nc1nnc(s1)S(N)(=O)=O',
                                 'niveau': 'Basis',
                                 'leitfrage': 'Warum ist Acetazolamid ein Brückenbeispiel zum Proteinlabor?',
                                 'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                                'Drehe anschließend das 3D-Modell.',
                                                'Vergleiche die Deskriptoren mit der Leitfrage.'],
                                 'frage': 'Welche Aussage passt am besten?',
                                 'antworten': ['Es besitzt mehrere polare Gruppen und Heteroatome.',
                                               'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                               'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                               'vorkommt.',
                                               'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                                 'richtig': 0,
                                 'feedback': 'Richtig: Es besitzt mehrere polare Gruppen und Heteroatome.',
                                 'erklaerung': 'Als Carboanhydrase-Hemmstoff eignet es sich für Struktur-Wirkungs- und '
                                               'Protein-Ligand-Bezüge.',
                                 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und '
                                                 'Kontext können dadurch nicht vollständig ersetzt werden.'},
                                {'name': 'Glycin',
                                 'smiles': 'NCC(=O)O',
                                 'niveau': 'Basis',
                                 'leitfrage': 'Warum ist Glycin die einfachste Aminosäure?',
                                 'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                                'Drehe anschließend das 3D-Modell.',
                                                'Vergleiche die Deskriptoren mit der Leitfrage.'],
                                 'frage': 'Welche Aussage passt am besten?',
                                 'antworten': ['Glycin besitzt Aminogruppe und Carboxylgruppe.',
                                               'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                               'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                               'vorkommt.',
                                               'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                                 'richtig': 0,
                                 'feedback': 'Richtig: Glycin besitzt Aminogruppe und Carboxylgruppe.',
                                 'erklaerung': 'Die Seitenkette ist nur ein Wasserstoffatom.',
                                 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und '
                                                 'Kontext können dadurch nicht vollständig ersetzt werden.'},
                                {'name': 'Alanin',
                                 'smiles': 'C[C@H](N)C(=O)O',
                                 'niveau': 'Basis',
                                 'leitfrage': 'Warum ist Alanin chiral?',
                                 'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                                'Drehe anschließend das 3D-Modell.',
                                                'Vergleiche die Deskriptoren mit der Leitfrage.'],
                                 'frage': 'Welche Aussage passt am besten?',
                                 'antworten': ['Das zentrale C-Atom trägt vier verschiedene Gruppen.',
                                               'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                               'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                               'vorkommt.',
                                               'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                                 'richtig': 0,
                                 'feedback': 'Richtig: Das zentrale C-Atom trägt vier verschiedene Gruppen.',
                                 'erklaerung': 'Alanin ist ein einfaches Beispiel für Chiralität in Biomolekülen.',
                                 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und '
                                                 'Kontext können dadurch nicht vollständig ersetzt werden.'},
                                {'name': 'Phenylalanin',
                                 'smiles': 'N[C@@H](Cc1ccccc1)C(=O)O',
                                 'niveau': 'Basis',
                                 'leitfrage': 'Was unterscheidet Phenylalanin von Glycin und Alanin?',
                                 'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                                'Drehe anschließend das 3D-Modell.',
                                                'Vergleiche die Deskriptoren mit der Leitfrage.'],
                                 'frage': 'Welche Aussage passt am besten?',
                                 'antworten': ['Die Seitenkette enthält einen aromatischen Ring.',
                                               'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                               'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                               'vorkommt.',
                                               'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                                 'richtig': 0,
                                 'feedback': 'Richtig: Die Seitenkette enthält einen aromatischen Ring.',
                                 'erklaerung': 'Aromatische hydrophobe Seitenketten sind wichtig für Proteinstruktur '
                                               'und Wechselwirkungen.',
                                 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und '
                                                 'Kontext können dadurch nicht vollständig ersetzt werden.'},
                                {'name': 'Adenin',
                                 'smiles': 'Nc1ncnc2ncnc12',
                                 'niveau': 'Basis',
                                 'leitfrage': 'Warum ist Adenin als Nukleobase interessant?',
                                 'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                                'Drehe anschließend das 3D-Modell.',
                                                'Vergleiche die Deskriptoren mit der Leitfrage.'],
                                 'frage': 'Welche Aussage passt am besten?',
                                 'antworten': ['Adenin ist eine stickstoffhaltige Nukleobase.',
                                               'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                               'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                               'vorkommt.',
                                               'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                                 'richtig': 0,
                                 'feedback': 'Richtig: Adenin ist eine stickstoffhaltige Nukleobase.',
                                 'erklaerung': 'Das heteroaromatische Ringsystem kann an Basenpaarung beteiligt sein.',
                                 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und '
                                                 'Kontext können dadurch nicht vollständig ersetzt werden.'},
                                {'name': 'Ribose',
                                 'smiles': 'O=CC(O)C(O)C(O)CO',
                                 'niveau': 'Basis',
                                 'leitfrage': 'Warum ist Ribose ein wichtiger RNA-Baustein?',
                                 'beobachten': ['Betrachte zuerst die 2D-Struktur.',
                                                'Drehe anschließend das 3D-Modell.',
                                                'Vergleiche die Deskriptoren mit der Leitfrage.'],
                                 'frage': 'Welche Aussage passt am besten?',
                                 'antworten': ['Ribose besitzt mehrere OH-Gruppen und ist stark polar.',
                                               'Die Summenformel allein erklärt alle Stoffeigenschaften.',
                                               'Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff '
                                               'vorkommt.',
                                               'Die räumliche Struktur spielt für Eigenschaften keine Rolle.'],
                                 'richtig': 0,
                                 'feedback': 'Richtig: Ribose besitzt mehrere OH-Gruppen und ist stark polar.',
                                 'erklaerung': 'In RNA ist Ribose Teil des Zucker-Phosphat-Rückgrats; hier wird die '
                                               'offenkettige Form vereinfacht gezeigt.',
                                 'modellgrenze': 'Die App liefert Modellwerte und Strukturhinweise. Experimente und '
                                                 'Kontext können dadurch nicht vollständig ersetzt werden.'},
                                {'name': 'Citronensäure',
                                 'smiles': 'O=C(O)CC(O)(CC(=O)O)C(=O)O',
                                 'niveau': 'Vertiefung',
                                 'leitfrage': 'Warum ist Citronensäure stark wasserfreundlich und als '
                                              'Stoffwechselprodukt interessant?',
                                 'beobachten': ['Suche die drei Carboxylgruppen.',
                                                'Suche die OH-Gruppe.',
                                                'Vergleiche TPSA und H-Brücken-Möglichkeiten.'],
                                 'frage': 'Welche Aussage passt am besten?',
                                 'antworten': ['Citronensäure besitzt mehrere polare Gruppen und kann stark mit Wasser '
                                               'wechselwirken.',
                                               'Citronensäure ist völlig unpolar.',
                                               'Citronensäure besitzt nur C-H-Bindungen.',
                                               'Citronensäure ist ein Protein.'],
                                 'richtig': 0,
                                 'feedback': 'Richtig: Mehrere Carboxylgruppen und eine OH-Gruppe sorgen für starke '
                                             'Wasserwechselwirkungen.',
                                 'erklaerung': 'Citronensäure ist noch überschaubar, zeigt aber bereits sehr deutlich, '
                                               'wie mehrere polare funktionelle Gruppen ein Molekül stark '
                                               'wasserfreundlich machen. Gleichzeitig stellt sie eine Brücke zum '
                                               'Stoffwechsel dar.',
                                 'modellgrenze': 'In biologischen Systemen liegen Carbonsäuren häufig teilweise oder '
                                                 'vollständig deprotoniert vor; diese einfache Darstellung bildet pH- '
                                                 'und Salzformen nicht vollständig ab.'}]}

# Gegenüberstellungen für das Modul „Isomerie“.
# Die Zuordnung ist absichtlich redundant in beide Richtungen, damit man von
# jedem Molekül des Paares aus den Vergleich starten kann.
ISOMER_PAIRS = {'Ethanol': 'Dimethylether',
 'Dimethylether': 'Ethanol',
 'Propanal': 'Aceton',
 'Aceton': 'Propanal',
 'Butan': 'Isobutan',
 'Isobutan': 'Butan',
 'Propansäure': 'Essigsäuremethylester',
 'Essigsäuremethylester': 'Propansäure',
 'D-Alanin': 'L-Alanin',
 'L-Alanin': 'D-Alanin',
 'Glucose offen': 'Fructose offen',
 'Fructose offen': 'Glucose offen',
 'Maleinsäure': 'Fumarsäure',
 'Fumarsäure': 'Maleinsäure'}


POLARITY_PAIRS = {'Wasser vs. Schwefelwasserstoff': {'a': 'Wasser',
                                    'b': 'Schwefelwasserstoff',
                                    'fehlvorstellung': 'Ähnliche Summenformel und ähnliche Molekülform bedeuten '
                                                       'ähnliche Eigenschaften.',
                                    'leitfrage': 'Warum ist Wasser bei Raumtemperatur flüssig, Schwefelwasserstoff '
                                                 'aber gasförmig?',
                                    'mini_aufgaben': ['Vergleiche die Molekülform beider Moleküle.',
                                                      'Suche mögliche Wasserstoffbrücken.',
                                                      'Erkläre, warum Wasser ein starkes Netzwerk zwischen den '
                                                      'Molekülen bilden kann.'],
                                    'kernaussage': 'Wasser bildet starke Wasserstoffbrücken; Schwefelwasserstoff hat '
                                                   'deutlich schwächere zwischenmolekulare Wechselwirkungen.',
                                    'experiment': 'Wasser ist bei Raumtemperatur flüssig; Schwefelwasserstoff ist '
                                                  'gasförmig.',
                                    'modellgrenze': 'Für kleine anorganische Moleküle sind logP/TPSA nur eingeschränkt '
                                                    'aussagekräftig. Hier zählt der experimentelle Bezug besonders '
                                                    'stark.'},
 'Methan vs. Hexan': {'a': 'Methan',
                      'b': 'Hexan',
                      'fehlvorstellung': 'Unpolar ist unpolar – daher müssten unpolare Stoffe ähnliche Eigenschaften '
                                         'besitzen.',
                      'leitfrage': 'Warum ist Methan gasförmig, Hexan aber flüssig, obwohl beide unpolar sind?',
                      'mini_aufgaben': ['Vergleiche die Molekülgröße.',
                                        'Betrachte die unpolare Oberfläche.',
                                        'Erkläre den Einfluss der Molekülgröße auf London-Dispersionskräfte.'],
                      'kernaussage': 'Auch unpolare Moleküle ziehen einander an. Größere Moleküle besitzen stärkere '
                                     'London-Kräfte.',
                      'experiment': 'Methan ist gasförmig; Hexan ist flüssig und kaum wasserlöslich.',
                      'modellgrenze': 'London-Kräfte werden nicht direkt berechnet, sondern nur über Größe und '
                                      'Oberfläche erschlossen.'},
 'Ethanol vs. 1-Hexanol': {'a': 'Ethanol',
                           'b': '1-Hexanol',
                           'fehlvorstellung': 'Alkohole sind wegen der OH-Gruppe immer gut wasserlöslich.',
                           'leitfrage': 'Warum ist Ethanol gut wasserlöslich, 1-Hexanol aber deutlich schlechter?',
                           'mini_aufgaben': ['Markiere in beiden Molekülen die OH-Gruppe.',
                                             'Vergleiche die Größe der unpolaren Kohlenwasserstoffreste.',
                                             'Begründe die unterschiedliche Wasserlöslichkeit.'],
                           'kernaussage': 'Die OH-Gruppe ermöglicht H-Brücken, aber bei längerer Kohlenstoffkette wird '
                                          'der unpolare Anteil immer wichtiger.',
                           'experiment': 'Ethanol ist mit Wasser mischbar; 1-Hexanol ist nur begrenzt wasserlöslich.',
                           'modellgrenze': 'Die Skala ist eine Orientierung und ersetzt keine Messung der '
                                           'Löslichkeit.'},
 'Essigsäure vs. Palmitinsäure': {'a': 'Essigsäure',
                                  'b': 'Palmitinsäure',
                                  'fehlvorstellung': 'Carbonsäuren sind wegen der COOH-Gruppe grundsätzlich gut '
                                                     'wasserlöslich.',
                                  'leitfrage': 'Warum ist Essigsäure gut wasserlöslich, Palmitinsäure aber kaum?',
                                  'mini_aufgaben': ['Suche die Carboxylgruppe in beiden Molekülen.',
                                                    'Vergleiche die unpolaren Molekülteile.',
                                                    'Erkläre, warum dieselbe funktionelle Gruppe nicht dieselbe '
                                                    'Stoffeigenschaft garantiert.'],
                                  'kernaussage': 'Eine polare Gruppe kann durch einen sehr großen unpolaren Rest in '
                                                 'ihrer Wirkung überlagert werden.',
                                  'experiment': 'Essigsäure ist gut wasserlöslich; Palmitinsäure ist in Wasser '
                                                'praktisch unlöslich.',
                                  'modellgrenze': 'pH-Effekte, Salzbildung, Kristallstruktur und Aggregation werden '
                                                  'nicht abgebildet.'},
 'Diethylether vs. Diethylenglycol': {'a': 'Diethylether',
                                      'b': 'Diethylenglycol',
                                      'fehlvorstellung': 'Ein Sauerstoffatom reicht aus, um Wasserlöslichkeit '
                                                         'zuverlässig vorherzusagen.',
                                      'leitfrage': 'Warum ist Diethylenglycol deutlich wasserfreundlicher als '
                                                   'Diethylether?',
                                      'mini_aufgaben': ['Zähle die O-Atome.',
                                                        'Suche OH-Gruppen.',
                                                        'Vergleiche H-Brücken-Donoren und -Akzeptoren.'],
                                      'kernaussage': 'Diethylether kann H-Brücken nur aufnehmen; Diethylenglycol kann '
                                                     'durch OH-Gruppen auch spenden und besitzt eine größere polare '
                                                     'Oberfläche.',
                                      'experiment': 'Diethylether ist nur begrenzt wasserlöslich; Diethylenglycol ist '
                                                    'sehr wasserfreundlich.',
                                      'modellgrenze': 'Toxizität und experimentelle Verwendung werden hier nicht '
                                                      'behandelt.'},
 'Methanol vs. Methanal': {'a': 'Methanol',
                           'b': 'Methanal',
                           'fehlvorstellung': 'Wenn beide Moleküle ein O-Atom besitzen, müssen sie ähnlich mit Wasser '
                                              'wechselwirken.',
                           'leitfrage': 'Warum unterscheiden sich Methanol und Methanal trotz ähnlicher Größe in ihren '
                                        'Wechselwirkungen?',
                           'mini_aufgaben': ['Suche OH-Gruppe bzw. Carbonylgruppe.',
                                             'Vergleiche H-Brücken-Donoren und -Akzeptoren.',
                                             'Erkläre den Unterschied zwischen Alkohol und Aldehyd.'],
                           'kernaussage': 'Methanol kann H-Brücken spenden und aufnehmen; Methanal kann nur aufnehmen '
                                          'und ist als Carbonylverbindung anders reaktiv.',
                           'experiment': 'Beide sind klein und wasserfreundlich, aber chemisch deutlich verschieden.',
                           'modellgrenze': 'Methanal liegt in Wasser teilweise hydratisiert vor; das einfache Modell '
                                           'bildet dies nicht ab.'},
 'Aceton vs. Essigsäure': {'a': 'Aceton',
                           'b': 'Essigsäure',
                           'fehlvorstellung': 'Eine C=O-Gruppe bestimmt die Eigenschaften allein.',
                           'leitfrage': 'Warum verhalten sich Aceton und Essigsäure trotz Carbonylgruppe '
                                        'unterschiedlich?',
                           'mini_aufgaben': ['Suche die Carbonylgruppe.',
                                             'Suche zusätzliche funktionelle Gruppen.',
                                             'Vergleiche H-Brücken und Säureeigenschaft.'],
                           'kernaussage': 'Aceton ist polar aprotisch; Essigsäure besitzt zusätzlich eine OH-Gruppe, '
                                          'kann H-Brücken spenden und ist sauer.',
                           'experiment': 'Beide sind gut wasserlöslich/mischbar, aber Essigsäure zeigt '
                                         'Säure-Base-Verhalten und starke H-Brücken.',
                           'modellgrenze': 'pH-Gleichgewichte und vollständige Säuredissoziation werden nicht '
                                           'berechnet.'}}


FUNCTIONAL_GROUPS = {
    "Alkohol": "[OX2H][CX4]",
    "Phenol": "[OX2H]c",
    "Aldehyd": "[CX3H1](=O)[#6,#1]",
    "Keton": "[#6][CX3](=O)[#6]",
    "Carbonsäure": "C(=O)[OX2H1]",
    "Ester": "C(=O)O[#6]",
    "Ether": "[#6][OD2][#6]",
    "Carbonsäureanhydrid": "C(=O)OC(=O)",
    "Amin": "[NX3;H2,H1,H0;!$(NC=O)]",
    "Amid": "C(=O)N",
    "Aromatischer Ring": "a",
    "Carbonylgruppe": "[CX3]=[OX1]",
}

current_report = {}

def mol_from_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Ungültiger SMILES-Code.")
    return mol

def make_3d(mol):
    mol_h = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    status = AllChem.EmbedMolecule(mol_h, params)
    if status != 0:
        AllChem.EmbedMolecule(mol_h, randomSeed=42)
    try:
        AllChem.MMFFOptimizeMolecule(mol_h, maxIters=300)
    except Exception:
        try:
            AllChem.UFFOptimizeMolecule(mol_h, maxIters=300)
        except Exception:
            pass
    return mol_h

def calc_properties(mol):
    return {
        "Summenformel": rdMolDescriptors.CalcMolFormula(mol),
        "Molmasse / g mol⁻¹": round(Descriptors.MolWt(mol), 2),
        "H-Brücken-Donoren": rdMolDescriptors.CalcNumHBD(mol),
        "H-Brücken-Akzeptoren": rdMolDescriptors.CalcNumHBA(mol),
        "logP (Modellwert)": round(Crippen.MolLogP(mol), 2),
        "TPSA / Å²": round(rdMolDescriptors.CalcTPSA(mol), 2),
        "rotierbare Bindungen": rdMolDescriptors.CalcNumRotatableBonds(mol),
        "Ringanzahl": rdMolDescriptors.CalcNumRings(mol),
        "schwere Atome": mol.GetNumHeavyAtoms(),
    }

def detect_functional_groups(mol):
    found=[]
    for name, smarts in FUNCTIONAL_GROUPS.items():
        patt = Chem.MolFromSmarts(smarts)
        if patt is not None and mol.HasSubstructMatch(patt):
            found.append(name)
    return sorted(set(found))

def mol_png_data_uri(mol, width=340, height=260):
    """2D-Struktur als eingebettetes PNG für kompakte HTML-Layouts."""
    img = Draw.MolToImage(mol, size=(width, height))
    buf = BytesIO()
    img.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    return f"data:image/png;base64,{b64}"

def show_2d(mol, width=360, height=260):
    display(HTML(f"<img src='{mol_png_data_uri(mol, width, height)}' style='max-width:100%;'>"))

def _style_for_3d(style_name):
    if style_name == "Kalottenmodell":
        return {"sphere": {"scale": 1.0}}
    if style_name == "Stabmodell":
        return {"stick": {"radius": 0.16}}
    # Default: Kugel-Stab
    return {"stick": {"radius": 0.12}, "sphere": {"scale": 0.25}}

def _style_js_for_3d(style_name):
    if style_name == "Kalottenmodell":
        return '{sphere:{scale:1.0}}'
    if style_name == "Stabmodell":
        return '{stick:{radius:0.16}}'
    if style_name == "Didaktische Polaritätsoberfläche":
        # Die Oberfläche selbst wird unten zusätzlich erzeugt.
        return '{stick:{radius:0.08}, sphere:{scale:0.18}}'
    return '{stick:{radius:0.12}, sphere:{scale:0.25}}'

def _is_polar_surface(style_name):
    return style_name == "Didaktische Polaritätsoberfläche"

def _custom_3dmol_iframe_html(molblock, width=520, height=380, style="Kugel-Stab"):
    # Schlanke 3Dmol.js-Ausgabe ohne py3Dmol-Wrapper.
    # v0.2: optional mit didaktischer Polaritätsoberfläche.
    div_id = "mol3d_" + uuid.uuid4().hex
    mol_js = json.dumps(molblock)
    style_js = _style_js_for_3d(style)
    add_surface_js = ""
    if _is_polar_surface(style):
        add_surface_js = """
      try {
        // Didaktische Oberfläche: elementbasierte VDW-Oberfläche, keine exakte ESP-Fläche.
        viewer.addSurface($3Dmol.SurfaceType.VDW, {opacity:0.72, colorscheme:'Jmol'}, {});
      } catch(surfaceErr) {
        console.log('Oberfläche konnte nicht erzeugt werden', surfaceErr);
      }
        """
    inner = f'''
<!doctype html>
<html>
<head>
<meta charset="utf-8">
<style>
  html, body {{ margin:0; padding:0; width:100%; height:100%; overflow:hidden; background:white; }}
  #{div_id} {{ width:100%; height:100%; }}
  .msg {{ font-family:Arial, sans-serif; font-size:12px; padding:8px; color:#666; }}
</style>
<script src="https://cdn.jsdelivr.net/npm/3dmol@2.5.5/build/3Dmol-min.js"></script>
</head>
<body>
<div id="{div_id}"><div class="msg">3D-Modell wird geladen …</div></div>
<script>
(function() {{
  const molblock = {mol_js};
  const styleObj = {style_js};
  function init(attempt) {{
    const el = document.getElementById('{div_id}');
    if (!el) return;
    if (typeof $3Dmol === 'undefined') {{
      if (attempt < 20) {{ window.setTimeout(function() {{ init(attempt+1); }}, 150); }}
      else {{ el.innerHTML = '<div class="msg">3Dmol.js konnte nicht geladen werden. Bitte erneut analysieren oder 3D-Engine wechseln.</div>'; }}
      return;
    }}
    el.innerHTML = '';
    try {{
      const viewer = $3Dmol.createViewer(el, {{backgroundColor:'white'}});
      viewer.addModel(molblock, 'sdf');
      viewer.setStyle({{}}, styleObj);
      {add_surface_js}
      viewer.zoomTo();
      viewer.render();
      window.addEventListener('resize', function() {{ viewer.resize(); viewer.render(); }});
    }} catch(e) {{
      el.innerHTML = '<div class="msg">3D-Anzeige fehlgeschlagen: ' + e + '</div>';
    }}
  }}
  init(0);
}})();
</script>
</body>
</html>
'''
    escaped = html.escape(inner, quote=True)
    return f'<iframe class="mol-lab-3d-frame" srcdoc="{escaped}" width="{width}" height="{height}" loading="eager" style="border:1px solid #ddd; border-radius:8px; background:white; max-width:100%;"></iframe>'

def _py3dmol_iframe_html(viewer, width=520, height=380):
    # Fallback-Ausgabe mit py3Dmol-eigenem HTML im iframe.
    try:
        inner = viewer._make_html()
        escaped = html.escape(inner, quote=True)
        return f'<iframe class="mol-lab-3d-frame" srcdoc="{escaped}" width="{width}" height="{height}" loading="eager" style="border:1px solid #ddd; border-radius:8px; background:white; max-width:100%;"></iframe>'
    except Exception as e:
        return f"<p style='color:#b00020'>3D-Anzeige konnte nicht erzeugt werden: {html.escape(str(e))}</p>"

def _viewer_for_molblock(molblock, width=520, height=380, style="Kugel-Stab"):
    viewer = py3Dmol.view(width=width, height=height)
    viewer.addModel(molblock, "sdf")
    viewer.setStyle(_style_for_3d(style))
    if _is_polar_surface(style):
        try:
            viewer.addSurface(py3Dmol.VDW, {"opacity":0.72, "colorscheme":"Jmol"})
        except Exception:
            pass
    viewer.setBackgroundColor("white")
    viewer.zoomTo()
    return viewer

def molblock_3d_html(molblock, width=520, height=380, style="Kugel-Stab", mode="custom"):
    if mode == "py3dmol":
        viewer = _viewer_for_molblock(molblock, width=width, height=height, style=style)
        return _py3dmol_iframe_html(viewer, width=width+20, height=height+30)
    return _custom_3dmol_iframe_html(molblock, width=width+20, height=height+30, style=style)

def mol3d_iframe_html(mol3d, width=520, height=380, style="Kugel-Stab", mode="custom"):
    mb = Chem.MolToMolBlock(mol3d)
    return molblock_3d_html(mb, width=width, height=height, style=style, mode=mode)

def show_3d(mol3d, width=520, height=380, style="Kugel-Stab", mode="custom"):
    display(HTML(mol3d_iframe_html(mol3d, width=width, height=height, style=style, mode=mode)))


def _label_bar_html(labels, active_index):
    """Kleine qualitative Skala als HTML."""
    cells=[]
    for i,lab in enumerate(labels):
        if i == active_index:
            cells.append(f"<span style='display:inline-block; padding:3px 7px; margin:1px; border-radius:10px; background:#e8f0fe; border:1px solid #5f7fcf; font-weight:bold;'>{html.escape(lab)}</span>")
        else:
            cells.append(f"<span style='display:inline-block; padding:3px 7px; margin:1px; border-radius:10px; background:#f6f6f6; border:1px solid #ddd; color:#666;'>{html.escape(lab)}</span>")
    return " ".join(cells)

def _formal_charge(mol):
    try:
        return sum(atom.GetFormalCharge() for atom in mol.GetAtoms())
    except Exception:
        return 0

def _hetero_heavy_count(mol):
    return sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() not in (1,6))

def qualitative_polarity_assessment(mol, props):
    """Didaktische Näherung für Polarität und Wasserlöslichkeit.

    Absichtlich keine exakte Stoffdatenvorhersage: Die Skalen übersetzen einfache
    RDKit-Modellgrößen in eine schülernahe Orientierung.
    """
    logp = float(props.get("logP (Modellwert)", 0) or 0)
    tpsa = float(props.get("TPSA / Å²", 0) or 0)
    hbd = int(props.get("H-Brücken-Donoren", 0) or 0)
    hba = int(props.get("H-Brücken-Akzeptoren", 0) or 0)
    heavy = max(int(props.get("schwere Atome", 1) or 1), 1)
    charge = abs(_formal_charge(mol))
    hetero = _hetero_heavy_count(mol)

    # Polaritätseindruck: eher Dichte polarer Oberfläche + H-Brücken-Möglichkeiten,
    # abgeschwächt durch deutlich lipophile Anteile.
    polar_score = 0.0
    polar_score += min(tpsa / 25.0, 3.0)
    polar_score += min(hbd * 0.8, 1.6)
    polar_score += min(hba * 0.25, 1.0)
    polar_score += min(charge * 1.4, 2.8)
    polar_score -= max(logp, 0) * 0.45
    polar_score -= max((heavy - 2*max(hetero,1)) / 18.0, 0)
    polar_score = max(polar_score, 0)

    if polar_score < 0.4:
        pol_i = 0
    elif polar_score < 1.0:
        pol_i = 1
    elif polar_score < 1.75:
        pol_i = 2
    elif polar_score < 2.8:
        pol_i = 3
    else:
        pol_i = 4

    # Wasserlöslichkeits-Tendenz: H-Brücken und TPSA begünstigen, logP und Größe
    # erschweren. Kleine polare Carbonyl-/Heteroatom-Moleküle erhalten einen Bonus.
    sol_score = 0.0
    sol_score += hbd * 1.2 + hba * 1.1 + tpsa / 25.0
    sol_score += min(charge * 3.0, 4.0)
    sol_score -= max(logp, 0) * 1.15
    sol_score -= max(heavy - 6, 0) * 0.25
    if heavy <= 5 and hba >= 1 and tpsa > 14:
        sol_score += 0.75
    sol_score = max(sol_score, 0)

    if sol_score < 0.45:
        sol_i = 0
    elif sol_score < 1.45:
        sol_i = 1
    elif sol_score < 2.35:
        sol_i = 2
    elif sol_score < 3.5:
        sol_i = 3
    else:
        sol_i = 4

    pol_labels = ["sehr schwach", "schwach", "mittel", "stark", "sehr stark"]
    sol_labels = ["kaum wasserlöslich", "begrenzt", "mäßig", "gut", "sehr gut"]

    reasons=[]
    if hbd > 0:
        reasons.append("kann H-Brücken spenden")
    if hba > 0:
        reasons.append("kann H-Brücken aufnehmen")
    if tpsa >= 70:
        reasons.append("große polare Oberfläche")
    elif tpsa >= 20:
        reasons.append("erkennbare polare Oberfläche")
    if logp >= 2.5:
        reasons.append("deutlich lipophiler Anteil")
    elif logp >= 0.8:
        reasons.append("unpolarer/lipophiler Anteil nimmt zu")
    if charge:
        reasons.append("formale Ladung vorhanden")
    if not reasons:
        reasons.append("kaum polare Gruppen erkennbar")

    return {
        "polarity_index": pol_i,
        "polarity_label": pol_labels[pol_i],
        "solubility_index": sol_i,
        "solubility_label": sol_labels[sol_i],
        "polarity_score": round(polar_score, 2),
        "solubility_score": round(sol_score, 2),
        "reason": "; ".join(reasons) + "."
    }

def _polarity_panel_html(mol, props):
    a = qualitative_polarity_assessment(mol, props)
    pol_labels = ["sehr schwach", "schwach", "mittel", "stark", "sehr stark"]
    sol_labels = ["kaum wasserlöslich", "begrenzt", "mäßig", "gut", "sehr gut"]
    return f"""
    <div style='margin-top:8px; padding:8px; border:1px solid #e0e0e0; border-radius:8px; background:#fbfbfb;'>
      <div style='font-size:0.95em; margin-bottom:5px;'><b>Didaktischer Polaritätseindruck:</b> {html.escape(a['polarity_label'])}</div>
      <div style='font-size:0.82em; margin-bottom:8px;'>{_label_bar_html(pol_labels, a['polarity_index'])}</div>
      <div style='font-size:0.95em; margin-bottom:5px;'><b>Wasserlöslichkeits-Tendenz:</b> {html.escape(a['solubility_label'])}</div>
      <div style='font-size:0.82em; margin-bottom:8px;'>{_label_bar_html(sol_labels, a['solubility_index'])}</div>
      <div style='font-size:0.82em; color:#555;'><b>Kurzbegründung:</b> {html.escape(a['reason'])}</div>
      <details style='font-size:0.78em; color:#555; line-height:1.32; margin-top:6px;'>
        <summary style='cursor:pointer;'>Näherung und Modellgrenzen anzeigen</summary>
        <div style='margin-top:5px;'>Diese Einschätzungen sind didaktische Näherungen. Sie werden aus einfachen Modellwerten wie logP, TPSA, Wasserstoffbrücken-Donoren/-Akzeptoren, Molekülgröße und formaler Ladung abgeleitet. Sie sollen Trends verständlich machen, ersetzen aber keine experimentellen Löslichkeitsdaten. Die tatsächliche Wasserlöslichkeit hängt zusätzlich von Temperatur, Ionisierungszustand, Konzentration, Molekülform und Wechselwirkungen mit dem Lösungsmittel ab.</div>
      </details>
    </div>
    """

def _surface_legend_html(style_name):
    if not _is_polar_surface(style_name):
        return ""
    return """
    <div style='font-size:0.78em; color:#555; line-height:1.3; margin-top:5px;'>
      <b>Didaktische Oberfläche:</b> elementbasierte Näherung, keine exakte elektrostatische Potentialfläche.
      Rot/blau/gelb/grün markieren heteroatomreiche, häufig polarere Bereiche; grau/weiß überwiegend C/H-Bereiche.
    </div>
    """

def _props_table_html(props):
    rows = []
    for k, v in props.items():
        rows.append(f"<tr><td style='padding:4px 8px; border-bottom:1px solid #eee;'>{html.escape(str(k))}</td><td style='padding:4px 8px; border-bottom:1px solid #eee; text-align:right;'><b>{html.escape(str(v))}</b></td></tr>")
    return "<table style='border-collapse:collapse; width:100%; font-size:0.92em;'>" + "".join(rows) + "</table>"

def _descriptor_note_html():
    return """
    <div style='font-size:0.78em; color:#555; line-height:1.32; margin-top:8px; border-top:1px solid #eee; padding-top:6px;'>
      <b>logP:</b> Modellwert für die Verteilung zwischen Fettphase und Wasser. Größer = eher lipophil, kleiner/negativ = eher hydrophil.<br>
      <b>TPSA:</b> topologische polare Oberfläche. Größer = mehr polare Oberfläche, oft stärkere Wechselwirkung mit Wasser/H-Brücken.
    </div>
    """

def display_structure_panel(mol, mol3d, props=None, groups=None, style="Kugel-Stab"):
    """Kompakte 2D-/3D-/Eigenschaften-Ansicht in einer Zeile, soweit Bildschirmbreite vorhanden ist."""
    png = mol_png_data_uri(mol, 300, 230)
    iframe = mol3d_iframe_html(mol3d, width=460, height=330, style=style, mode=render_mode_dd.value)
    props_html = _props_table_html(props or {}) if props else ""
    groups_html = ", ".join(groups or []) if groups else "keine aus der einfachen Liste erkannt"
    display(HTML(f"""
    <div style="display:flex; flex-wrap:wrap; gap:14px; align-items:flex-start; margin-top:8px;">
      <div style="flex:0 0 315px; max-width:100%;">
        <h3 style="margin:0 0 6px 0;">2D-Struktur</h3>
        <div style="border:1px solid #ddd; border-radius:8px; background:white; padding:8px; text-align:center;">
          <img src="{png}" style="max-width:100%; height:auto;">
        </div>
      </div>
      <div style="flex:0 0 500px; max-width:100%;">
        <h3 style="margin:0 0 6px 0;">3D-Modell</h3>
        {iframe}
        {_surface_legend_html(style)}
      </div>
      <div style="flex:1 1 330px; min-width:300px; max-width:520px;">
        <h3 style="margin:0 0 6px 0;">Eigenschaften</h3>
        <div style="border:1px solid #ddd; border-radius:8px; background:white; padding:8px;">
          {props_html}
          {_polarity_panel_html(mol, props) if props else ""}
          {_descriptor_note_html() if props else ""}
        </div>
        <h3 style="margin:12px 0 6px 0;">Funktionelle Gruppen</h3>
        <div style="border:1px solid #ddd; border-radius:8px; background:white; padding:8px;">{html.escape(groups_html)}</div>
      </div>
    </div>
    """))


def _structure_card_html(card, style="Kugel-Stab", width3d=330, height3d=255):
    """Kompakte HTML-Karte mit 2D- und 3D-Struktur für Vergleichsansichten."""
    mol = mol_from_smiles(card["smiles"])
    mol3d = make_3d(mol)
    png = mol_png_data_uri(mol, 230, 175)
    iframe = mol3d_iframe_html(mol3d, width=width3d, height=height3d, style=style, mode=render_mode_dd.value)
    return f"""
    <div style="flex:1 1 430px; min-width:390px; max-width:520px; border:1px solid #ddd; border-radius:10px; padding:10px; background:#fff;">
      <h3 style="margin-top:0;">{html.escape(card['name'])}</h3>
      <p style="margin:4px 0 8px 0;"><b>SMILES:</b> <code>{html.escape(card['smiles'])}</code></p>
      <div style="display:flex; flex-wrap:wrap; gap:10px; align-items:flex-start;">
        <div style="flex:0 0 240px; text-align:center;">
          <b>2D</b><br>
          <img src="{png}" style="max-width:100%; height:auto;">
        </div>
        <div style="flex:1 1 340px; min-width:320px;">
          <b>3D</b><br>
          {iframe}
          {_surface_legend_html(style)}
        </div>
      </div>
    </div>
    """

def display_isomer_comparison(card_a, card_b, style="Kugel-Stab"):
    """Isomerenpaar nebeneinander mit Strukturen und Eigenschaften darstellen."""
    mol_a = mol_from_smiles(card_a["smiles"])
    mol_b = mol_from_smiles(card_b["smiles"])
    props_a = calc_properties(mol_a)
    props_b = calc_properties(mol_b)
    groups_a = detect_functional_groups(mol_a)
    groups_b = detect_functional_groups(mol_b)
    display(HTML("""
    <h3>Isomeren-Vergleich</h3>
    <p>Beide Moleküle werden direkt gegenübergestellt. Achte besonders darauf, ob die <b>Summenformel gleich</b> ist, aber funktionelle Gruppen, räumliche Anordnung und Eigenschaften unterschiedlich sein können.</p><p style='font-size:0.92em;color:#555'><b>Hinweis:</b> Manche berechneten Modell-Deskriptoren, z. B. logP oder TPSA, können bei bestimmten Isomeren gleich ausfallen. Das bedeutet nicht, dass die Stoffe identisch sind; Reaktivität, Dipolmoment, Siedepunkt oder biologische Wirkung können trotzdem deutlich verschieden sein.</p>
    """))
    display(HTML("<div style='display:flex; flex-wrap:wrap; gap:16px; align-items:flex-start;'>" +
                 _structure_card_html(card_a, style=style) +
                 _structure_card_html(card_b, style=style) +
                 "</div>"))
    keys = ["Summenformel", "Molmasse / g mol⁻¹", "H-Brücken-Donoren", "H-Brücken-Akzeptoren", "logP (Modellwert)", "TPSA / Å²", "rotierbare Bindungen", "Ringanzahl"]
    rows=[]
    for k in keys:
        rows.append({"Eigenschaft": k, card_a["name"]: props_a.get(k,""), card_b["name"]: props_b.get(k,"")})
    qa = qualitative_polarity_assessment(mol_a, props_a)
    qb = qualitative_polarity_assessment(mol_b, props_b)
    rows.append({"Eigenschaft":"Didaktischer Polaritätseindruck", card_a["name"]: qa["polarity_label"], card_b["name"]: qb["polarity_label"]})
    rows.append({"Eigenschaft":"Wasserlöslichkeits-Tendenz", card_a["name"]: qa["solubility_label"], card_b["name"]: qb["solubility_label"]})
    rows.append({"Eigenschaft":"erkannte funktionelle Gruppen", card_a["name"]:", ".join(groups_a) or "keine erkannt", card_b["name"]:", ".join(groups_b) or "keine erkannt"})
    display(pd.DataFrame(rows))
    display(HTML(_descriptor_note_html()))
    display(HTML("""
    <p><b>Arbeitsauftrag:</b> Formuliere in einem Satz, welche Strukturänderung den wichtigsten Unterschied zwischen den beiden Molekülen ausmacht. Überlege anschließend, welche Stoffeigenschaft dadurch besonders beeinflusst werden könnte.</p>
    """))

def _display_py3dmol(viewer, width=520, height=400):
    display(HTML(_py3dmol_iframe_html(viewer, width=width, height=height)))

def show_conf(mol, conf_id, title="Konformer", style="Kugel-Stab", width=360, height=300):
    mb = Chem.MolToMolBlock(mol, confId=conf_id)
    display(HTML(f"<b>{title}</b><br>" + molblock_3d_html(mb, width=width, height=height, style=style, mode=render_mode_dd.value)))

def conf_iframe_html(mol, conf_id, title="Konformer", style="Kugel-Stab", width=360, height=300):
    mb = Chem.MolToMolBlock(mol, confId=conf_id)
    frame = molblock_3d_html(mb, width=width, height=height, style=style, mode=render_mode_dd.value)
    return f"<div style='flex:1 1 380px; min-width:330px; max-width:520px;'><b>{html.escape(title)}</b><br>" + frame + "</div>"

def display_conformers_side_by_side(mol, conformer_list, style="Kugel-Stab", max_n=4):
    """Konformere horizontal nebeneinander als getrennte, unabhängig drehbare 3D-Fenster."""
    boxes=[]
    for i,(cid,e) in enumerate(conformer_list[:max_n]):
        boxes.append(conf_iframe_html(mol, cid, f"Konformer {i+1}, ΔE = {e} kcal/mol", style=style, width=340, height=280))
    display(HTML("<div style='display:flex; flex-wrap:wrap; gap:14px; align-items:flex-start;'>" + "".join(boxes) + "</div>"))

def interpret_properties(props):
    parts=[]
    logp=props.get("logP (Modellwert)",0)
    tpsa=props.get("TPSA / Å²",0)
    hbd=props.get("H-Brücken-Donoren",0)
    hba=props.get("H-Brücken-Akzeptoren",0)
    rot=props.get("rotierbare Bindungen",0)
    if hbd>0: parts.append("Das Molekül kann Wasserstoffbrücken spenden.")
    else: parts.append("Das Molekül kann keine Wasserstoffbrücken spenden.")
    if hba>0: parts.append("Es kann Wasserstoffbrücken aufnehmen.")
    if tpsa < 20: parts.append("Die berechnete polare Oberfläche ist klein; das spricht für eher unpolare Eigenschaften.")
    elif tpsa < 80: parts.append("Die berechnete polare Oberfläche ist mittelgroß; polare und unpolare Bereiche können nebeneinander vorkommen.")
    else: parts.append("Die berechnete polare Oberfläche ist groß; deutliche polare Bereiche sind zu erwarten.")
    if logp < 0: parts.append("Der logP-Wert spricht eher für hydrophile Eigenschaften.")
    elif logp < 3: parts.append("Der logP-Wert spricht für ein gemischtes Verhältnis zwischen hydrophilen und lipophilen Anteilen.")
    else: parts.append("Der logP-Wert spricht für ein eher lipophiles Molekül.")
    if rot >= 5: parts.append("Mehrere rotierbare Bindungen machen das Molekül vergleichsweise flexibel.")
    elif rot >= 1: parts.append("Das Molekül besitzt einige bewegliche Einfachbindungen.")
    else: parts.append("Das Molekül ist relativ starr.")
    return " ".join(parts)

def generate_conformers(smiles, n=60):
    """Einfache RDKit-Konformersuche mit Energiestufen-Filter.

    Viele mehrfach erzeugte Konformere landen nach der Optimierung im gleichen
    Minimum. Für den Unterricht ist es hilfreicher, unterschiedliche
    Energieniveaus zu zeigen statt fünfmal ΔE = 0.
    """
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    params.pruneRmsThresh = -1.0
    try:
        params.useRandomCoords = True
    except Exception:
        pass
    ids = list(AllChem.EmbedMultipleConfs(mol, numConfs=n, params=params))
    results=[]
    props = AllChem.MMFFGetMoleculeProperties(mol)
    for cid in ids:
        try:
            if props is not None:
                ff = AllChem.MMFFGetMoleculeForceField(mol, props, confId=cid)
            else:
                ff = AllChem.UFFGetMoleculeForceField(mol, confId=cid)
            ff.Minimize(maxIts=500)
            e = ff.CalcEnergy()
        except Exception:
            ff = AllChem.UFFGetMoleculeForceField(mol, confId=cid)
            ff.Minimize(maxIts=500)
            e = ff.CalcEnergy()
        results.append((cid,e))
    if not results:
        return mol, [], []
    mine=min(e for _,e in results)
    rel=[(cid, e-mine) for cid,e in results]
    rel.sort(key=lambda x:x[1])

    # unterschiedliche Energieniveaus auswählen; sonst würden bei Butan z.B.
    # viele äquivalente anti-Konformere mit ΔE = 0 erscheinen.
    unique=[]
    for cid,e in rel:
        if all(abs(e - eu) > 0.08 for _, eu in unique):
            unique.append((cid,e))
        if len(unique) >= 6:
            break
    rel_round=[(cid, round(e,2)) for cid,e in rel]
    unique_round=[(cid, round(e,2)) for cid,e in unique]
    return mol, unique_round, rel_round

def card_for(module, name):
    for c in MOLECULES[module]:
        if c["name"] == name:
            return c
    return None

def report_text(rep):
    lines=[]
    lines.append(f"MOL_LAB DIDACTIC – Kurzanalyse")
    lines.append(f"Molekül: {rep.get('name','')}")
    lines.append(f"SMILES: {rep.get('smiles','')}")
    lines.append("")
    lines.append("Eigenschaften:")
    for k,v in rep.get("props",{}).items(): lines.append(f"- {k}: {v}")
    lines.append("")
    lines.append("Funktionelle Gruppen: " + (", ".join(rep.get('groups',[])) or "keine erkannt"))
    if rep.get('assessment'):
        lines.append("Didaktischer Polaritätseindruck: " + rep['assessment'].get('polarity_label',''))
        lines.append("Wasserlöslichkeits-Tendenz: " + rep['assessment'].get('solubility_label',''))
        lines.append("Kurzbegründung: " + rep['assessment'].get('reason',''))
    lines.append("")
    lines.append("Leitfrage: " + rep.get('leitfrage',''))
    lines.append("Modell-Deutung: " + rep.get('interpretation',''))
    lines.append("Erklärung: " + rep.get('erklaerung',''))
    lines.append("Modellgrenze: " + rep.get('modellgrenze',''))
    return "\n".join(lines)

# Oberfläche
module_dd = widgets.Dropdown(options=list(MOLECULES.keys()), description="Modul:", layout=widgets.Layout(width="520px"))
example_dd = widgets.Dropdown(description="Beispiel:", layout=widgets.Layout(width="520px"))
custom_smiles = widgets.Text(value="", placeholder="optional eigener SMILES-Code, z.B. CCO", description="SMILES:", layout=widgets.Layout(width="700px"))
use_custom = widgets.Checkbox(value=False, description="eigenen SMILES verwenden")
style_dd = widgets.ToggleButtons(
    options=["Kugel-Stab", "Stabmodell", "Kalottenmodell", "Didaktische Polaritätsoberfläche"],
    value="Kugel-Stab",
    description="3D:",
    button_style="",
    layout=widgets.Layout(width="100%")
)

render_mode_dd = widgets.Dropdown(
    options=[
        ("3Dmol.js leicht (empfohlen)", "custom"),
        ("py3Dmol iFrame (Fallback)", "py3dmol")
    ],
    value="custom",
    description="3D-Engine:",
    layout=widgets.Layout(width="360px")
)
run_btn = widgets.Button(description="Molekül analysieren", button_style="success")
answer_radio = widgets.RadioButtons(options=[], description="Antwort:", layout=widgets.Layout(width="900px"))
check_btn = widgets.Button(description="Antwort prüfen", button_style="info")
explain_btn = widgets.Button(description="Erklärung anzeigen")
report_btn = widgets.Button(description="Analyse als Text anzeigen")
conf_btn = widgets.Button(description="Konformere berechnen", button_style="warning")
compare_btn = widgets.Button(description="Isomerenpaar vergleichen", button_style="primary")
polar_pair_dd = widgets.Dropdown(
    options=list(POLARITY_PAIRS.keys()),
    description="Vergleich:",
    layout=widgets.Layout(width="620px")
)
polar_compare_btn = widgets.Button(description="Polaritäts-/Löslichkeitsvergleich", button_style="primary")
reset3d_btn = widgets.Button(description="3D-Anzeige neu aufbauen")
out = widgets.Output()
feedback_out = widgets.Output()
report_out = widgets.Output()


def refresh_examples(*args):
    module = module_dd.value
    names = [c["name"] for c in MOLECULES[module]]
    # In Colab hängen Dropdowns gelegentlich, wenn Optionen direkt ersetzt werden.
    # Daher erst leeren, dann neu setzen und den ersten Wert aktiv auswählen.
    example_dd.options = []
    example_dd.options = tuple(names)
    if names:
        example_dd.value = names[0]

    # Vergleichssteuerung bewusst nur dort anzeigen, wo sie didaktisch geführt ist.
    polar_pair_dd.layout.display = "" if module == "Polarität und Löslichkeit" else "none"
    polar_compare_btn.layout.display = "" if module == "Polarität und Löslichkeit" else "none"
    compare_btn.layout.display = "" if module == "Isomerie" else "none"
    out.clear_output(); feedback_out.clear_output(); report_out.clear_output()

refresh_examples()
module_dd.observe(refresh_examples, names="value")

def analyze(_=None):
    global current_report
    with out:
        clear_output()
        gc.collect()
        module=module_dd.value
        if use_custom.value:
            smiles=custom_smiles.value.strip()
            if not smiles:
                print("Bitte einen SMILES-Code eingeben oder das Häkchen entfernen."); return
            card={"name":"Eigenes Molekül", "smiles":smiles, "leitfrage":"Welche Strukturmerkmale könnten die Stoffeigenschaften beeinflussen?", "beobachten":["Suche polare Gruppen.","Achte auf die Molekülgröße.","Vergleiche logP, TPSA und H-Brücken."], "frage":"Welche Aussage ist bei eigenen Molekülen besonders wichtig?", "antworten":["Die berechneten Werte sind Modellwerte und müssen fachlich geprüft werden.","SMILES-Eingaben sind immer experimentelle Messwerte.","Die Summenformel allein reicht immer aus.","Große Moleküle sind in dieser App immer zuverlässig berechnet."], "richtig":0, "feedback":"Richtig: Die App gibt Modellhinweise, keine endgültigen Messwerte.", "erklaerung":"Eigene SMILES-Eingaben sind für kleine bis mittelkleine Moleküle gedacht. Prüfe immer, ob die Struktur chemisch sinnvoll ist.", "modellgrenze":"Für große, geladene oder ungewöhnliche Moleküle ist diese einfache RDKit-Analyse nur eingeschränkt geeignet."}
        else:
            card=card_for(module, example_dd.value)
            smiles=card["smiles"]
        try:
            mol=mol_from_smiles(smiles)
        except Exception as e:
            print("Fehler:", e); return
        props=calc_properties(mol)
        groups=detect_functional_groups(mol)
        assessment=qualitative_polarity_assessment(mol, props)
        interp=interpret_properties(props)
        display(HTML(f"<h2>{card['name']}</h2><p><b>SMILES:</b> <code>{smiles}</code></p>"))
        display(HTML(f"<h3>Leitfrage</h3><p>{card.get('leitfrage','')}</p>"))
        display(HTML("<h3>Beobachte</h3><ul>" + "".join(f"<li>{x}</li>" for x in card.get('beobachten',[])) + "</ul>"))
        display_structure_panel(mol, make_3d(mol), props=props, groups=groups, style=style_dd.value)
        display(HTML("<h3>Erste Modell-Deutung</h3><p>" + interp + "</p>"))
        answer_radio.options=card.get("antworten",[])
        answer_radio.value=answer_radio.options[0] if answer_radio.options else None
        display(HTML("<h3>Vermute</h3><p>" + card.get('frage','') + "</p>"))
        display(answer_radio, widgets.HBox([check_btn, explain_btn, report_btn]))
        current_report={"name":card["name"], "smiles":smiles, "card":card, "props":props, "groups":groups, "assessment":assessment, "leitfrage":card.get("leitfrage",""), "interpretation":interp, "erklaerung":card.get("erklaerung",""), "modellgrenze":card.get("modellgrenze","")}
    feedback_out.clear_output(); report_out.clear_output()

def check_answer(_=None):
    with feedback_out:
        clear_output()
        if not current_report: return
        card=current_report["card"]
        selected=answer_radio.value
        idx=list(answer_radio.options).index(selected)
        if idx == card.get("richtig",0):
            display(HTML("<p style='color:green'><b>Richtig.</b> " + card.get("feedback","") + "</p>"))
        else:
            right=card.get("antworten",[])[card.get("richtig",0)]
            display(HTML("<p style='color:#b00020'><b>Noch nicht ganz.</b> Die beste Antwort ist: <b>" + right + "</b></p><p>" + card.get("feedback","") + "</p>"))

def explain(_=None):
    with feedback_out:
        clear_output()
        if not current_report: return
        display(HTML("<h3>Erklärung</h3><p>" + current_report.get("erklaerung","") + "</p><h4>Grenze des Modells</h4><p>" + current_report.get("modellgrenze","") + "</p>"))

def show_report(_=None):
    with report_out:
        clear_output()
        if not current_report: return
        txt=report_text(current_report)
        display(widgets.Textarea(value=txt, layout=widgets.Layout(width="100%", height="260px")))

def do_conformers(_=None):
    with report_out:
        clear_output()
        if not current_report:
            print("Bitte zuerst ein Molekül analysieren."); return
        smiles=current_report["smiles"]
        mol0 = Chem.MolFromSmiles(smiles)
        if mol0 is None or mol0.GetNumHeavyAtoms() > 35:
            print("Konformer-Suche nur für kleine bis mittelgroße Moleküle empfohlen."); return
        display(HTML("<h3>Konformere: einfache RDKit/MMFF-Suche</h3><p>Relative Energien sind Modellwerte in kcal/mol. Mehrfach gefundene gleiche Minima werden zusammengefasst. Bei kleinen Molekülen wie Butan oder 1,2-Dichlorethan sind oft nur zwei echte Minima sinnvoll: anti und gauche. Ekliptische Formen sind Übergangslagen und verschwinden bei einer normalen Geometrieoptimierung.</p>"))
        mol, unique_rel, all_rel = generate_conformers(smiles, n=120)
        if not unique_rel:
            print("Keine Konformere erzeugt."); return
        display(pd.DataFrame([{
            "Energieniveau": i+1,
            "confId": cid,
            "relative Energie / kcal mol⁻¹": e
        } for i,(cid,e) in enumerate(unique_rel[:6])]))
        if len(unique_rel) <= 2:
            display(HTML("<p><b>Hinweis:</b> Dass hier nur wenige Konformere erscheinen, ist meist kein Fehler: Nach der Optimierung bleiben nur stabile Minima übrig. Für Butan sind anti und gauche die entscheidenden Minima; ekliptische Formen wären keine stabilen Konformere.</p>"))
        display(HTML("<h4>Vergleichsansicht</h4><p>Die Konformere werden nebeneinander in getrennten 3D-Fenstern angezeigt und können unabhängig voneinander gedreht werden.</p>"))
        display_conformers_side_by_side(mol, unique_rel, style=style_dd.value, max_n=4)



def _comparison_assessment_label(mol, props):
    a = qualitative_polarity_assessment(mol, props)
    pol_labels = ["sehr schwach", "schwach", "mittel", "stark", "sehr stark"]
    sol_labels = ["kaum wasserlöslich", "begrenzt", "mäßig", "gut", "sehr gut"]
    return pol_labels[a["polarity_index"]], sol_labels[a["solubility_index"]]


def display_polarity_comparison(pair_key, style="Kugel-Stab"):
    """Kuratierter Vergleich im Modul Polarität und Löslichkeit."""
    pair = POLARITY_PAIRS[pair_key]
    card_a = card_for("Polarität und Löslichkeit", pair["a"])
    card_b = card_for("Polarität und Löslichkeit", pair["b"])
    if not card_a or not card_b:
        print("Das Vergleichspaar wurde in der Moleküldatenbank nicht gefunden.")
        return

    mol_a = mol_from_smiles(card_a["smiles"]); mol_b = mol_from_smiles(card_b["smiles"])
    props_a = calc_properties(mol_a); props_b = calc_properties(mol_b)
    groups_a = detect_functional_groups(mol_a); groups_b = detect_functional_groups(mol_b)
    pol_a, sol_a = _comparison_assessment_label(mol_a, props_a)
    pol_b, sol_b = _comparison_assessment_label(mol_b, props_b)

    display(HTML(f"""
    <h3>Vergleichspaar: {html.escape(pair_key)}</h3>
    <div style='background:#fff7e6; border-left:4px solid #f0ad4e; padding:8px 10px; margin:8px 0;'>
      <b>Typische Fehlvorstellung:</b> {html.escape(pair.get('fehlvorstellung',''))}
    </div>
    <p><b>Leitfrage:</b> {html.escape(pair.get('leitfrage',''))}</p>
    """))

    display(HTML("<div style='display:flex; flex-wrap:wrap; gap:14px; align-items:flex-start;'>" +
                 _structure_card_html(card_a, style=style, width3d=310, height3d=235) +
                 _structure_card_html(card_b, style=style, width3d=310, height3d=235) +
                 "</div>"))

    rows=[]
    keys = ["Summenformel", "Molmasse / g mol⁻¹", "H-Brücken-Donoren", "H-Brücken-Akzeptoren", "logP (Modellwert)", "TPSA / Å²", "Rotierbare Bindungen"]
    for k in keys:
        rows.append({"Eigenschaft": k, card_a["name"]: props_a.get(k,""), card_b["name"]: props_b.get(k,"")})
    rows.append({"Eigenschaft": "Polaritätseindruck", card_a["name"]: pol_a, card_b["name"]: pol_b})
    rows.append({"Eigenschaft": "Wasserlöslichkeits-Tendenz", card_a["name"]: sol_a, card_b["name"]: sol_b})
    rows.append({"Eigenschaft": "erkannte Gruppen", card_a["name"]: ", ".join(groups_a) or "keine erkannt", card_b["name"]: ", ".join(groups_b) or "keine erkannt"})
    display(pd.DataFrame(rows))
    display(HTML(_descriptor_note_html()))

    display(HTML("<h4>Mini-Aufgaben</h4><ol>" + "".join(f"<li>{html.escape(x)}</li>" for x in pair.get('mini_aufgaben',[])) + "</ol>"))
    if pair.get("experiment"):
        display(HTML(f"""
        <details style='margin:8px 0;'>
          <summary><b>Experimenteller Bezug anzeigen</b></summary>
          <div style='font-size:0.92em; color:#333; padding:6px 0;'>{html.escape(pair.get('experiment',''))}</div>
        </details>
        """))
    display(HTML(f"""
    <details style='margin:8px 0;'>
      <summary><b>Lösungsvorschlag / Kernaussage anzeigen</b></summary>
      <div style='font-size:0.92em; color:#333; padding:6px 0;'><b>Kernaussage:</b> {html.escape(pair.get('kernaussage',''))}<br>
      <b>Modellgrenze:</b> {html.escape(pair.get('modellgrenze',''))}</div>
    </details>
    """))


def do_polarity_comparison(_=None):
    with report_out:
        clear_output()
        if module_dd.value != "Polarität und Löslichkeit":
            print("Der Polaritäts-/Löslichkeitsvergleich ist nur im Modul Polarität und Löslichkeit aktiv.")
            return
        key = polar_pair_dd.value
        if not key:
            print("Bitte ein Vergleichspaar wählen.")
            return
        display_polarity_comparison(key, style=style_dd.value)

def do_isomer_comparison(_=None):
    with report_out:
        clear_output()
        if module_dd.value != "Isomerie":
            print("Der Vergleich ist nur im Modul Isomerie aktiv.")
            return
        if use_custom.value:
            print("Für eigene SMILES ist noch kein automatischer Paarvergleich definiert.")
            return
        name_a = example_dd.value
        name_b = ISOMER_PAIRS.get(name_a)
        if not name_b:
            print("Für dieses Beispiel ist noch kein Isomerenpaar hinterlegt.")
            return
        card_a = card_for("Isomerie", name_a)
        card_b = card_for("Isomerie", name_b)
        if not card_a or not card_b:
            print("Das hinterlegte Vergleichspaar wurde nicht gefunden.")
            return
        display_isomer_comparison(card_a, card_b, style=style_dd.value)

def reset_3d(_=None):
    """Sicherheitsknopf: 3D-iframes entfernen und letzte Analyse neu aufbauen."""
    try:
        display(Javascript("""
        document.querySelectorAll('iframe.mol-lab-3d-frame').forEach(function(el){ el.remove(); });
        """))
    except Exception:
        pass
    feedback_out.clear_output(); report_out.clear_output()
    gc.collect()
    analyze()

run_btn.on_click(analyze)
check_btn.on_click(check_answer)
explain_btn.on_click(explain)
report_btn.on_click(show_report)
conf_btn.on_click(do_conformers)
compare_btn.on_click(do_isomer_comparison)
polar_compare_btn.on_click(do_polarity_comparison)
reset3d_btn.on_click(reset_3d)

display(HTML('<h1>MOL_LAB DIDACTIC <span style="font-size:0.55em;color:#666">v0.3</span></h1><p>Wähle ein Modul und ein Beispiel oder gib einen eigenen SMILES-Code ein.</p>'))
controls = widgets.VBox([
    widgets.HTML("<b>1. Lernmodul und Beispiel auswählen</b>"),
    module_dd,
    example_dd,
    widgets.HTML("<b>2. 3D-Darstellung</b>"),
    style_dd,
    render_mode_dd,
    widgets.HTML("<b>3. Optional: eigenes Molekül</b>"),
    use_custom,
    custom_smiles,
    widgets.HTML("<b>4. Analyse und geführte Vergleiche</b>"),
    polar_pair_dd,
    widgets.HBox([run_btn, conf_btn, compare_btn, polar_compare_btn, reset3d_btn]),
    widgets.HTML("<small>Hinweis: Die Vergleichspaare sind bewusst kuratiert. Freie SMILES-Eingaben sind zum Erkunden gedacht, ersetzen aber keine fachliche Prüfung.</small>")
])
display(controls, out, feedback_out, report_out)


## 3. Hinweise für weitere Versionen

In v0.3 neu:
- kuratierte Vergleichspaare im Modul **Polarität und Löslichkeit**
- Mini-Aufgaben und optionaler experimenteller Bezug bei ausgewählten Paaren
- zusätzliche didaktische Beispiele: u. a. Schwefelwasserstoff, Hexan, 1-Hexanol, Palmitinsäure, Diethylenglycol, Maleinsäure/Fumarsäure, Salicylsäure, Vanillin, Citronensäure
- 3D-Darstellungsarten als direkte Schaltflächen statt Dropdown

Bewusst weiterhin nicht enthalten:
- kein Lernmanagement-System
- keine komplexen Biomoleküle wie ATP oder Acetyl-CoA
- keine xTB-/ESP-Berechnung; die Polaritätsoberfläche bleibt eine didaktische Näherung

Vorbereitet für später:
- einklappbare 3D-Großansicht
- sehr einfacher SchülerInnen-Protokollexport
- optionale Erweiterungsaufgaben wie Ascorbinsäure oder D-/L-Glycerinaldehyd
